<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/2026_06_11_DROID_dataset_v60.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### DROID: A Large-Scale In-The-Wild Robot Manipulation Dataset

![](https://droid-dataset.github.io/droid/assets/index/droid_teaser.jpg)

This Colab demonstrates how to load and visualize samples from the DROID dataset. Please also check out our [dataset visualizer](https://droid-dataset.github.io/dataset.html) to explore the dataset.

You can download the full dataset (1.7TB) using:
```
gsutil -m cp -r gs://gresearch/robotics/droid <your_local_path>
```

If you'd like to download an example version of the dataset with 100 episodes first (2GB), run:
```
gsutil -m cp -r gs://gresearch/robotics/droid_100 <your_local_path>
```

If you want to use DROID for policy training, please check out our [policy training repo](https://github.com/droid-dataset/droid_policy_learning).

### 环境配置与初始化

In [ ]:
# @title 安装 Python 依赖库

!pip install mediapy pyrender trimesh PyOpenGL-accelerate pybullet polyscope yourdfpy

In [ ]:
import os
import sys

repo_url = "https://github.com/yangyi02/droid.git"
repo_dir = "/content/droid"

# 如果没有克隆过，就 clone；如果已经存在，就 pull 最新代码
if not os.path.exists(repo_dir):
    !git clone {repo_url}
else:
    %cd {repo_dir}
    !git pull
    %cd /content

# 把仓库目录加入环境变量，这样就能直接 import 了
if repo_dir not in sys.path:
    sys.path.append(repo_dir)

# 举个例子：
# from utils.geometry import unproject_points_np

%load_ext autoreload
%autoreload 2

In [ ]:
# @title 导入通用 Python 库

import copy
import os
import sys
import gc
import glob
import cv2
import h5py
import importlib.util
import json
from matplotlib import cm
import matplotlib.pyplot as plt
import mediapy as media
import numpy as np
from PIL import Image, ImageDraw
import plotly.graph_objects as go
import plotly.io as pio
import polyscope as ps
import pybullet as p
import pybullet_data
import pyrender
import random
from scipy.spatial.transform import Rotation as R, Slerp
import trimesh
import tensorflow_datasets as tfds
from tqdm import tqdm
import torch
import torch.nn.functional as F
import torch.optim as optim
import yourdfpy

os.environ['PYOPENGL_PLATFORM'] = 'egl'
pio.renderers.default = 'colab'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
# @title 克隆 Github 仓库 (CoTracker)

%cd /content

!git clone https://github.com/facebookresearch/co-tracker.git
sys.path.append("/content/co-tracker")

from cotracker.predictor import CoTrackerPredictor
print("✅ [3/4] CoTracker3 稠密点追踪模块 导入成功！")

In [ ]:
# @title 下载与初始化 CoTracker3 模型

from cotracker.predictor import CoTrackerPredictor

WEIGHTS_URL = "https://huggingface.co/facebook/cotracker3/resolve/main/scaled_offline.pth"
os.makedirs("/content/co-tracker/weights", exist_ok=True)
weights_path = os.path.join("/content/co-tracker/weights/", "cotracker3_offline.pth")

if not os.path.exists(weights_path):
    !wget -q {WEIGHTS_URL} -O {weights_path}

cotracker_model = CoTrackerPredictor(checkpoint=weights_path)
cotracker_model = cotracker_model.to(device)

print("✅ 模型加载成功！现在可以继续运行后续的 2D 跟踪代码了。")

In [ ]:
# @title 下载 Robotiq URDF 与 3D 物理资产

# 1. 克隆包含定制版 Franka + Robotiq 模型的官方仓库 (强行指定 data 分支)
!git clone -b data https://github.com/NVlabs/PointWorld.git

In [ ]:
# @title 下载并解析 DROID 数据集元数据 (JSON)

root_path = "/content/droid_raw/1.0.1"
base_url = "https://huggingface.co/KarlP/droid/resolve/main"

# 🗑️ 移除 cam2base_extrinsic_superset.json
files = ["intrinsics.json", "camera_serials.json", "episode_id_to_path.json", "keep_ranges_1_0_1.json", "cam2base_extrinsic_superset.json"]

os.makedirs(root_path, exist_ok=True)
for f in files:
    os.system(f"wget -q -nc -P {root_path} {base_url}/{f}")

def load_json(name):
    with open(os.path.join(root_path, name), 'r') as f: return json.load(f)

serials_db, id_to_path = load_json(files[1]), load_json(files[2])
keep_ranges = load_json(files[3])
extrinsics_db = load_json(files[4])

# 🌟 解锁封印：现在的 valid_ids 只受限于基本元数据，直接打通全部数据！
valid_ids = sorted(set(serials_db.keys()) & set(id_to_path.keys()))
print(f"✅ 全局元数据准备完毕！匹配到 {len(valid_ids)} 个 Episode。")

# 🌟 新增：计算包含官方外参的 Episode 集合
episodes_with_ext = set(valid_ids) & set(extrinsics_db.keys())
print(f"   📸 其中包含官方预标定外参的 Episode 数量为: {len(episodes_with_ext)} 个。")

# 只采样有初始相机标定的数据
valid_ids = sorted(set(serials_db.keys()) & set(id_to_path.keys()) & set(extrinsics_db.keys()))
print(f"✅ 全局元数据准备完毕！匹配到包含官方预标定外参的 Episode 共 {len(valid_ids)} 个。")

In [ ]:
# @title 已处理的 episode
valid_ids = [
    "GuptaLab+553d1bd5+2023-04-30-16h-07m-27s",
    "GuptaLab+553d1bd5+2023-05-28-16h-48m-13s",
    "GuptaLab+553d1bd5+2023-05-28-18h-22m-39s",
    "ILIAD+5e938e3b+2023-07-20-11h-50m-51s",
    "ILIAD+7ae1bcff+2023-06-04-20h-13m-12s",
    "IPRL+edf28ef3+2024-01-01-10h-36m-18s",
    "IRIS+7dfa2da3+2023-04-17-16h-24m-37s",
    "IRIS+7dfa2da3+2023-05-11-14h-12m-57s",
    "IRIS+7dfa2da3+2023-06-01-13h-48m-16s",
    "PennPAL+c5f808b7+2023-06-15-17h-12m-39s",
    "RAIL+d027f2ae+2023-11-04-14h-49m-32s",
    "REAL+4f8ca688+2023-08-29-15h-01m-27s",
    "REAL+75b7b0f9+2023-06-23-16h-46m-08s",
    "REAL+de601749+2023-12-19-18h-19m-27s",
    "TRI+52ca9b6a+2023-12-05-15h-19m-45s",
    "TRI+52ca9b6a+2024-01-23-16h-53m-54s",
]

### 数据准备

In [ ]:
# @title 下载 DROID Episode 并初始化全局状态池

def download_episode(episode_id, root_path, id_to_path, serials_db, keep_ranges_db):
    """一站式下载视频，并按层级分类初始化 scene_constants (元数据终极聚合版)"""
    # id_to_path[episode_id] 本身就是类似 "RPL/failure/2023-11-30-18h-16m-00s" 的相对路径
    relative_path = id_to_path[episode_id]
    episode_path = os.path.join(root_path, relative_path)

    if not os.path.exists(episode_path):
        os.makedirs(episode_path, exist_ok=True)
        os.system(f"gsutil -m -q cp -r \"gs://gresearch/robotics/droid_raw/1.0.1/{relative_path}/*\" \"{episode_path}/\"")
        print(f"  ⬇️ 成功下载数据: {episode_id}")
    else:
        print(f"  ⏭️ 数据已存在，跳过下载。")

    cam_info = serials_db[episode_id]
    wrist_serial = cam_info.get('wrist_cam_serial')

    # 🗑️ 彻底脱离 intrinsics_db 依赖，直接从串口数据中提取全部机位
    valid_cams = sorted(set(cam_info.values()))

    # 🌟 修复：完美还原官方 JSON 里的 absolute bucket path 格式
    base_prefix = "gs://xembodiment_data/r2d2/r2d2-data-full/"
    episode_key = f"{base_prefix}{relative_path}/recordings/MP4--{base_prefix}{relative_path}/trajectory.h5"

    valid_indices = None

    if episode_key in keep_ranges_db:
        ranges = keep_ranges_db[episode_key]
        indices = []
        for start, end in ranges:
            indices.extend(range(start, end))
        valid_indices = np.array(indices)
        print(f"  ✂️ 已预加载动作区间，共标记 {len(valid_indices)} 帧为有效关键帧。")
    else:
        # 为了防范极端情况，如果真找不到（尽管概率很小），保留全量帧
        print(f"  ⚠️ 未找到 {episode_id} 的 Idle 过滤信息，将默认保留全量帧。")

    # 🌟 极简封箱：结构化分为 meta, robot, camera 三大核心模块
    return {
        'meta': {
            'episode_id': episode_id,
            'episode_path': episode_path,
            'wrist_serial': wrist_serial,
            'valid_indices': valid_indices  # <--- 完美注入！
        },
        'robot': {},  # 预留位，供后续运动学解析填入
        'camera': {
            cam: {
                'baseline': 0.063 if cam == wrist_serial else 0.120,
            } for cam in valid_cams  # 字典推导式优雅注入所有相机
        }
    }

# ================= 主流程 =================
episode_id = random.choice(valid_ids)
print(f"🎯 选定处理的 Episode: {episode_id}")

# 🌟 移除了 intrinsics_db 传参
scene_constants = download_episode(episode_id, root_path, id_to_path, serials_db, keep_ranges)

print(f"✅ scene_constants 初始化就绪！")
print(f"   - 腕部相机: {scene_constants['meta']['wrist_serial']}")
print(f"   - 已加载机位: {list(scene_constants['camera'].keys())}")
if scene_constants['meta']['valid_indices'] is not None:
    print(f"   - 动作有效帧数: {len(scene_constants['meta']['valid_indices'])}")

In [ ]:
# @title 从 GCS Bucket 加载深度

def load_stage1_camera_data(scene_constants, bucket_prefix="gs://dm-tapnet/mv-tap/droid/depth"):
    episode_id = scene_constants['meta']['episode_id']
    local_cache_dir = f"/content/droid_depth_cache/{episode_id}"
    os.makedirs(local_cache_dir, exist_ok=True)

    print(f"  ☁️ 免授权极速加载: 绕过目录遍历，直接拉取 {episode_id} 的确切文件...")

    # ==========================================
    # 🌟 1. 匿名下载并解析 Robot 数据
    # ==========================================
    robot_gcs_path = f"{bucket_prefix}/{episode_id}/robot.npz"
    robot_local_path = os.path.join(local_cache_dir, "robot.npz")

    # 显式下载确切的单文件，绝不使用通配符 (*)
    os.system(f"gsutil cp '{robot_gcs_path}' '{local_cache_dir}/' > /dev/null 2>&1")

    # 强制读取 (如果文件不存在直接报错 FileNotFoundError)
    robot_data = np.load(robot_local_path, allow_pickle=True)

    # 安全读取 robot.npz 中的变量
    for k in ['joint_positions', 'gripper_positions', 'T_cam_ee_init', 'T_ee_base_all']:
        if k in robot_data:
            scene_constants['robot'][k] = robot_data[k]

    if 'valid_indices' in robot_data:
        scene_constants['meta']['valid_indices'] = robot_data['valid_indices']
    if 'wrist_serial' in robot_data:
        scene_constants['meta']['wrist_serial'] = str(robot_data['wrist_serial'].item())

    wrist_serial = scene_constants['meta'].get('wrist_serial')
    print(f"    ✅ 成功匿名拉取并挂载 robot.npz")

    # ==========================================
    # 🌟 2. 匿名并发下载并解析各相机数据
    # ==========================================
    # 所有相机共有的基础文件
    base_cam_files = [
        "video_left.mp4", "video_right.mp4",
        "video_left_raw.mp4", "video_right_raw.mp4",
        "raw_depth.npz", "calibration.npz"
    ]

    for cam in scene_constants['camera']:
        local_cam_dir = os.path.join(local_cache_dir, cam)
        os.makedirs(local_cam_dir, exist_ok=True)

        # 如果是腕部相机，追加专属的夹爪提纯工件
        cam_files = list(base_cam_files)
        if cam == wrist_serial:
            cam_files.extend([
                "original_raw_depth.npz",
                "gripper_mask.npz",
                "gripper_depth.npz"
            ])

        # 拼接出极其精确的 GCS 路径
        gcs_files = [f"'{bucket_prefix}/{episode_id}/{cam}/{fname}'" for fname in cam_files]
        gcs_files_str = " ".join(gcs_files)

        # 核心黑科技：一股脑丢给 gsutil，它会自动并发 Get，完美绕过 List 校验
        os.system(f"gsutil -m cp {gcs_files_str} '{local_cam_dir}/' > /dev/null 2>&1")

        # --- 强制读取 4 路 MP4 视频 ---
        video_keys = {
            "video_rgb": "video_left.mp4",
            "video_right": "video_right.mp4",
            "video_raw_rgb": "video_left_raw.mp4",
            "video_raw_right": "video_right_raw.mp4",
        }
        for mem_key, filename in video_keys.items():
            vid_path = os.path.join(local_cam_dir, filename)
            if os.path.exists(vid_path):
                scene_constants['camera'][cam][mem_key] = media.read_video(vid_path)

        # --- 读取深度图并还原单位 (mm -> meters) ---
        depth_path = os.path.join(local_cam_dir, "raw_depth.npz")
        if os.path.exists(depth_path):
            scene_constants['camera'][cam]['raw_depth'] = np.load(depth_path)['depth'].astype(np.float32) / 1000.0

        # --- 读取腕部专属工件 ---
        orig_depth_path = os.path.join(local_cam_dir, "original_raw_depth.npz")
        if os.path.exists(orig_depth_path):
            scene_constants['camera'][cam]['original_raw_depth'] = np.load(orig_depth_path)['depth'].astype(np.float32) / 1000.0

        mask_path = os.path.join(local_cam_dir, "gripper_mask.npz")
        if os.path.exists(mask_path):
            scene_constants['camera'][cam]['sam_real_masks'] = np.load(mask_path)['mask']

        grip_depth_path = os.path.join(local_cam_dir, "gripper_depth.npz")
        if os.path.exists(grip_depth_path):
            scene_constants['camera'][cam]['empirical_gripper_depth'] = np.load(grip_depth_path)['depth'].astype(np.float32) / 1000.0

        # --- 强制重构标定参数字典 ---
        calib_path = os.path.join(local_cam_dir, "calibration.npz")
        if os.path.exists(calib_path):
            calib_npz = np.load(calib_path)
            scene_constants['camera'][cam]['K_mat'] = calib_npz['K_calib_left']
            scene_constants['camera'][cam]['baseline'] = float(calib_npz['baseline'])

            scene_constants['camera'][cam]['zed_calibration'] = {
                'calibrated': {
                    'K': calib_npz['K_calib_left'], 'disto': calib_npz['disto_calib_left'],
                    'K_right': calib_npz['K_calib_right'], 'disto_right': calib_npz['disto_calib_right']
                },
                'raw': {
                    'K': calib_npz['K_raw_left'], 'disto': calib_npz['disto_raw_left'],
                    'K_right': calib_npz['K_raw_right'], 'disto_right': calib_npz['disto_raw_right']
                }
            }

        print(f"    ✅ 成功匿名拉取相机 {cam} 的全部数据")

    return scene_constants

# ================= 主流程 =================
scene_constants = load_stage1_camera_data(scene_constants)

In [ ]:
# @title 从 GCS Bucket 加载外参

def load_stage2_extrinsics_json(scene_constants, bucket_prefix="gs://dm-tapnet/mv-tap/droid/extrinsics"):
    episode_id = scene_constants['meta']['episode_id']
    local_cache_dir = f"/content/droid_extrinsics_cache/{episode_id}"
    os.makedirs(local_cache_dir, exist_ok=True)

    print(f"\n🚀 [Stage 2] 启动闭环外参 (Extrinsics) 加载引擎 (JSON 分布式版)...")
    print(f"  ☁️ 免授权极速加载: 直接拉取 {episode_id} 各机位的 extrinsics.json...")

    cam_keys = list(scene_constants['camera'].keys())
    scene_state = {}
    success_count = 0
    missing_count = 0

    print("  " + "="*60)
    for cam in cam_keys:
        cam_dir = os.path.join(local_cache_dir, cam)
        os.makedirs(cam_dir, exist_ok=True)

        # 🌟 1. 拼接直击 final 版本 JSON 的 GCS 路径
        gcs_path = f"{bucket_prefix}/{episode_id}/{cam}/extrinsics.json"
        local_path = os.path.join(cam_dir, "extrinsics.json")

        # 显式下载单相机的 JSON
        os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")

        # 🌟 2. 解析 JSON 并还原为 NumPy 张量大盘
        if os.path.exists(local_path):
            with open(local_path, 'r') as f:
                ext_data = json.load(f)

            # JSON 中的嵌套列表无缝转回 NumPy (统一转为 float32 以节省显存)
            base_ext = np.array(ext_data['base_extrinsic'], dtype=np.float32)
            full_ext = np.array(ext_data['extrinsics'], dtype=np.float32)
            is_wrist = ext_data.get('is_wrist', False)

            # 组装进 scene_state
            scene_state[cam] = {
                'base_extrinsic': base_ext,
                'extrinsics': full_ext,
                'is_wrist': is_wrist
            }

            # 维度与异常值健康检查
            has_nan = np.isnan(full_ext).any()
            nan_flag = "❌ 含 NaN" if has_nan else "✅ 数据健康"
            wrist_flag = "🦾 腕部相机" if is_wrist else "🎥 环境相机"

            print(f"    ✅ 相机 [{cam}] JSON 挂载成功 | {wrist_flag} | Shape: {full_ext.shape} | {nan_flag}")
            success_count += 1
        else:
            print(f"    ⚠️ 相机 [{cam}] 缺失 | 未能在 GCS 中找到对应的 extrinsics.json")
            missing_count += 1

    print("  " + "="*60)

    # 🌟 3. 大盘总结
    print(f"\n📊 JSON Extrinsics 最终质检大盘: 预期 {len(cam_keys)} 个 | 成功 {success_count} 个 | 缺失 {missing_count} 个")

    return scene_state

# ================= 主流程 =================
try:
    # 替换调用新的 JSON 加载器
    scene_state = load_stage2_extrinsics_json(scene_constants)
    print("\n🎯 JSON 数据集加载完成！时序轨迹已全部转换为 NumPy 矩阵。")
except Exception as e:
    print(f"\n❌ 加载失败: {e}")

In [ ]:
# @title 🎯 3D 视觉几何核心大一统算子库

def decode_disparity_np(disp, fx, baseline):
    """将原始视差转化为物理深度 (NumPy)"""
    z = np.zeros_like(disp)
    valid_mask = disp > 0  # 过滤无效视差
    z[valid_mask] = (fx * baseline) / disp[valid_mask]
    return z

def decode_disparity_pt(disp, fx, baseline):
    """将原始视差转化为物理深度 (PyTorch 完美梯度版)"""
    z = torch.zeros_like(disp)
    valid_mask = disp > 0  # 过滤无效视差
    z[valid_mask] = (fx * baseline) / disp[valid_mask]
    return z

def unproject_points_np(u, v, z, K, T_cam2world=None):
    """纯粹的 3D 射线反投影算子，只处理合法的物理深度 Z (NumPy)"""
    x_cam = (u - K[0, 2]) * z / K[0, 0]
    y_cam = (v - K[1, 2]) * z / K[1, 1]

    pts_cam = np.stack([x_cam, y_cam, z, np.ones_like(z)], axis=0)

    if T_cam2world is None:
        return pts_cam[:3, :].T
    return (T_cam2world @ pts_cam)[:3, :].T

def unproject_points_pt(u, v, z, K, T_cam2world=None):
    """纯粹的 3D 射线反投影算子，完美保持梯度穿透 (PyTorch)"""
    x_cam = (u - K[0, 2]) * z / K[0, 0]
    y_cam = (v - K[1, 2]) * z / K[1, 1]

    pts_cam = torch.stack([x_cam, y_cam, z, torch.ones_like(z)], dim=0)

    if T_cam2world is None:
        return pts_cam[:3, :].T
    return (T_cam2world @ pts_cam)[:3, :].T

def project_points_np(pts_world, K, T_cam2world):
    """将 3D 世界点阵拍回 2D 像素平面 (NumPy)"""
    T_world2cam = np.linalg.inv(T_cam2world)
    pts_homo = np.hstack([pts_world, np.ones((len(pts_world), 1))]).T
    pts_cam = T_world2cam @ pts_homo

    z_cam = pts_cam[2, :]
    u = np.zeros_like(pts_cam[0, :])
    v = np.zeros_like(pts_cam[1, :])

    valid_mask = z_cam > 0
    u[valid_mask] = (pts_cam[0, valid_mask] / z_cam[valid_mask]) * K[0, 0] + K[0, 2]
    v[valid_mask] = (pts_cam[1, valid_mask] / z_cam[valid_mask]) * K[1, 1] + K[1, 2]

    return u, v, z_cam

def project_points_pt(pts_world, K, T_cam2world):
    """将 3D 世界点阵拍回 2D 像素平面，完美保留计算图 (PyTorch)"""
    T_world2cam = torch.linalg.inv(T_cam2world)
    pts_homo = torch.cat([pts_world, torch.ones((len(pts_world), 1), device=pts_world.device)], dim=1).T
    pts_cam = T_world2cam @ pts_homo

    z_cam = pts_cam[2, :]
    u = torch.zeros_like(pts_cam[0, :])
    v = torch.zeros_like(pts_cam[1, :])

    valid_mask = z_cam > 0
    u[valid_mask] = (pts_cam[0, valid_mask] / z_cam[valid_mask]) * K[0, 0] + K[0, 2]
    v[valid_mask] = (pts_cam[1, valid_mask] / z_cam[valid_mask]) * K[1, 1] + K[1, 2]

    return u, v, z_cam

In [ ]:
# @title 点云可视化函数

def unproject_to_3d(depth, color_img, K_mat, T_cam2world=None, min_depth=0., max_depth=1.5):
    """纯粹的几何升维算子：输入已校准的深度，专心做空间截断与反投影"""
    # 🌟 1. 空间屏蔽：用最符合直觉的物理阈值过滤
    mask = (depth > min_depth) & (depth < max_depth)
    v, u = np.where(mask)

    # 🌟 2. 几何升维：调用底层原生算子
    if T_cam2world is None:
        T_cam2world = np.eye(4)
    pts_world = unproject_points_np(u, v, depth[mask], K_mat, T_cam2world)

    return pts_world, color_img[mask]

def show_plotly_point_cloud(pts, cols, title="3D Point Cloud", max_points=150000, eye_pos=(-1.5, -1.5, 1.0)):
    """交互式点云渲染 (保持极简瘦身版不变)"""
    idx = np.random.permutation(len(pts))[:max_points]
    p, c = pts[idx], cols[idx]

    go.Figure(
        data=[go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers',
                           marker=dict(size=1.5, color=[f'rgb({r},{g},{b})' for r, g, b in c]))],
        layout=go.Layout(
            title=title, margin=dict(l=0, r=0, b=0, t=40), height=500, showlegend=False,
            scene=dict(aspectmode='data', camera=dict(eye=dict(x=eye_pos[0], y=eye_pos[1], z=eye_pos[2])))
        )
    ).show(renderer="colab")

In [ ]:
# @title 3D 单帧融合点云与交互式可视化

def render_fused_point_cloud(scene_constants, scene_state, frame_idx=0,
                             max_render_points=150000, eye_pos=(-1.2, -1.2, 0.8), use_tint=False):

    # 🌟 1. 拥抱新架构：直接提取 camera 层级的 key，并利用 sorted 保证多机位合并顺序的绝对一致性
    camera_ids = sorted(scene_constants['camera'].keys())

    # 🌟 2. 调色板保留：严格保留原版染色矩阵 (绿、红、蓝)，用于 Debug 模式下区分机位
    tint_colors = np.array([[0, 50, 0], [50, 0, 0], [0, 0, 50]])

    fused_points, fused_colors = [], []

    # 🌟 3. 语义复苏遍历：使用清晰的命名
    for idx, cam_id in enumerate(camera_ids):
        cam_data = scene_constants['camera'][cam_id]
        cam_state = scene_state[cam_id]

        # 🌟 直接提取原始深度图 (彻底剥离 scale 和 shift)
        raw_depth = cam_data['raw_depth'][frame_idx].astype(np.float32)

        # 🌟 直接传入 raw_depth 提取 3D 点 (依赖 unproject_to_3d 内部的深度范围截断)
        points_3d, colors_rgb = unproject_to_3d(
            raw_depth,
            cam_data['video_rgb'][frame_idx],
            cam_data['K_mat'],
            T_cam2world=cam_state['extrinsics'][frame_idx]
        )

        # 染色处理
        if use_tint:
            colors_rgb = np.clip(colors_rgb.astype(int) + tint_colors[idx % len(tint_colors)], 0, 255).astype(np.uint8)

        fused_points.append(points_3d)
        fused_colors.append(colors_rgb)

    # 🌟 4. 极致清爽的堆叠与渲染
    show_plotly_point_cloud(
        pts=np.vstack(fused_points),
        cols=np.vstack(fused_colors),
        title=f"Fused Point Cloud (Frame {frame_idx})" + (" 🎨 [Tinted Debug Mode]" if use_tint else ""),
        max_points=max_render_points,
        eye_pos=eye_pos
    )

In [ ]:
# @title 2D Mask 全视场联合体检函数

import inspect

def render_multiview_mask_inspection(scene_constants, scene_state, pb_renderer, frame_idx=0):
    """
    全视场三联屏监视器 (支持新老两代 Renderer 无缝切换 + 数据结构极致解耦版)
    """
    camera_ids = list(scene_constants['camera'].keys())
    wrist_cam = scene_constants['meta']['wrist_serial']

    # 🌟 1. 提取姿态数据
    joint_angles = scene_constants['robot']['joint_positions'][frame_idx]
    gripper_state = scene_constants['robot']['gripper_positions'][frame_idx]

    # 🌟 2. 智能路由 A：检测引擎是否支持控制夹爪
    sig = inspect.signature(pb_renderer.update_robot_pose)
    pb_renderer.update_robot_pose(joint_angles, gripper_state=gripper_state)

    # 开启多联屏画布
    fig, axes = plt.subplots(1, len(camera_ids), figsize=(12, 3))
    if len(camera_ids) == 1: axes = [axes]

    fig.suptitle(f"Multi-View Segmentation Mask Inspection (Frame {frame_idx})",
                 fontsize=20, fontweight='bold', y=1.05)

    for i, cam_id in enumerate(camera_ids):

        # 🌟 核心进化：彻底干掉 if/else 特判！
        # 无论你是环境相机还是腕部相机，直接从统一大盘中抽出当前帧的 4x4 绝对位姿
        extrinsics = scene_state[cam_id]['extrinsics'][frame_idx]

        intrinsics = scene_constants['camera'][cam_id]['K_mat']
        img_rgb = scene_constants['camera'][cam_id]['video_rgb'][frame_idx].copy()
        h_img, w_img = img_rgb.shape[:2]

        # 🌟 3. 智能路由 B：检测引擎用的是哪个渲染方法
        robot_mask = pb_renderer.render_mask(extrinsics, intrinsics, w_img, h_img) > 0

        # 亮绿色半透明叠加
        overlay = img_rgb.copy()
        overlay[robot_mask] = [50, 255, 50]
        blended_img = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

        # 绘制
        cam_type = "Wrist Camera" if cam_id == wrist_cam else "External Camera"
        axes[i].imshow(blended_img)
        axes[i].set_title(f"[{cam_type}]\nCam ID: {cam_id}", fontsize=15)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# @title 通用数字孪生影棚

class PyBulletRenderer_Robotiq:
    def __init__(self, ghost_urdf="PointWorld/assets/franka_description/franka_panda_robotiq_2f85_og.urdf"):
        if p.isConnected(): p.disconnect()
        p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())

        if importlib.util.find_spec('eglRendererPlugin'):
            p.loadPlugin(importlib.util.find_spec('eglRendererPlugin').origin, "_eglRendererPlugin")

        # 实体 (The Body)：原厂极瘦机械臂
        self.robot_id = p.loadURDF("franka_panda/panda.urdf", useFixedBase=True)
        self.arm_joints = [i for i in range(p.getNumJoints(self.robot_id)) if "panda_joint" in p.getJointInfo(self.robot_id, i)[1].decode('utf-8') and p.getJointInfo(self.robot_id, i)[2] != p.JOINT_FIXED]

        self.hidden_robot_links = []
        for i in range(-1, p.getNumJoints(self.robot_id)):
            name = p.getBodyInfo(self.robot_id)[0].decode('utf-8') if i == -1 else p.getJointInfo(self.robot_id, i)[12].decode('utf-8')
            if "hand" in name or "finger" in name:
                p.changeVisualShape(self.robot_id, i, rgbaColor=[0, 0, 0, 0])
                self.hidden_robot_links.append(i)

        # 替身 (The Ghost)：携带夹爪的 PointWorld 机械臂
        self.ghost_id = p.loadURDF(ghost_urdf, useFixedBase=True)
        self.ghost_arm_joints = [i for i in range(p.getNumJoints(self.ghost_id)) if "panda_joint" in p.getJointInfo(self.ghost_id, i)[1].decode('utf-8') and p.getJointInfo(self.ghost_id, i)[2] != p.JOINT_FIXED]

        self.gripper_joints = []
        self.gripper_signs = []

        for i in range(p.getNumJoints(self.ghost_id)):
            info = p.getJointInfo(self.ghost_id, i)
            joint_name = info[1].decode('utf-8')
            joint_type = info[2]

            if joint_type != p.JOINT_FIXED and "panda_joint" not in joint_name:
                self.gripper_joints.append(i)
                base_sign = -1 if "right" in joint_name else 1
                if "inner_finger" in joint_name or "follower" in joint_name or "finger_tip" in joint_name:
                    self.gripper_signs.append(base_sign * -1)
                else:
                    self.gripper_signs.append(base_sign)

        self.hidden_ghost_links = []
        for i in range(-1, p.getNumJoints(self.ghost_id)):
            name = p.getBodyInfo(self.ghost_id)[0].decode('utf-8') if i == -1 else p.getJointInfo(self.ghost_id, i)[12].decode('utf-8')
            if "panda_link" in name:
                p.changeVisualShape(self.ghost_id, i, rgbaColor=[0, 0, 0, 0])
                self.hidden_ghost_links.append(i)

    def _get_projection_matrix(self, intrinsics, width, height):
        fx, fy, cx, cy = intrinsics[0, 0], intrinsics[1, 1], intrinsics[0, 2], intrinsics[1, 2]
        near, far = 0.01, 10.0
        return [2.0 * fx / width, 0.0, 0.0, 0.0,
                0.0, 2.0 * fy / height, 0.0, 0.0,
                1.0 - 2.0 * cx / width, 2.0 * cy / height - 1.0, (far + near) / (near - far), -1.0,
                0.0, 0.0, 2.0 * far * near / (near - far), 0.0]

    def update_robot_pose(self, joint_angles, gripper_state=None, gripper_width_offset=0.08):
        for i, angle in zip(self.arm_joints, joint_angles):
            p.resetJointState(self.robot_id, i, angle)
        for i, angle in zip(self.ghost_arm_joints, joint_angles):
            p.resetJointState(self.ghost_id, i, angle)

        if gripper_state is not None and len(self.gripper_joints) > 0:
            raw_val = gripper_state[0] if isinstance(gripper_state, (list, np.ndarray)) else gripper_state
            raw_val = np.clip(raw_val, 0.0, 1.0)
            max_urdf_radian = 0.8028
            angle = (raw_val * max_urdf_radian) - gripper_width_offset

            for i, sign in zip(self.gripper_joints, self.gripper_signs):
                p.resetJointState(self.ghost_id, i, angle * sign)
        p.performCollisionDetection()

    # 🌟 核心融合修改：支持一键切分本体或夹爪深度
    def render_depth(self, extrinsics, intrinsics, width, height, only_gripper=False):
        cam_pos, target_pos = extrinsics[:3, 3], extrinsics[:3, 3] + extrinsics[:3, 2]
        view_matrix = p.computeViewMatrix(cam_pos, target_pos, -extrinsics[:3, 1])
        proj_matrix = self._get_projection_matrix(intrinsics, width, height)
        _, _, _, depth_buffer, seg_buffer = p.getCameraImage(
            width, height, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
            renderer=p.ER_BULLET_HARDWARE_OPENGL, flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX
        )
        metric_depth = 0.1 / (10.0 - 9.99 * np.reshape(depth_buffer, (height, width)))
        depth_valid = metric_depth < 9.9

        if only_gripper:
            seg_array = np.reshape(seg_buffer, (height, width)).astype(np.int32)
            obj_ids = seg_array & 0xFFFFFF
            depth_valid = depth_valid & (obj_ids == self.ghost_id)

        return np.where(depth_valid, metric_depth, 0.0)

    def render_mask(self, extrinsics, intrinsics, width, height):
        cam_pos, target_pos = extrinsics[:3, 3], extrinsics[:3, 3] + extrinsics[:3, 2]
        view_matrix = p.computeViewMatrix(cam_pos, target_pos, -extrinsics[:3, 1])
        proj_matrix = self._get_projection_matrix(intrinsics, width, height)
        _, _, _, _, seg_buffer = p.getCameraImage(
            width, height, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
            renderer=p.ER_BULLET_HARDWARE_OPENGL, flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX
        )
        seg_array = np.reshape(seg_buffer, (height, width)).astype(np.int32)
        obj_ids, link_ids = seg_array & 0xFFFFFF, (seg_array >> 24) - 1
        valid_robot = (obj_ids == self.robot_id) & ~np.isin(link_ids, self.hidden_robot_links)
        valid_ghost = (obj_ids == self.ghost_id) & ~np.isin(link_ids, self.hidden_ghost_links)
        return valid_robot | valid_ghost

In [ ]:
# @title 2D 机械臂分割可视化 (双极态极限对比验证版)

def inspect_gripper_extremes(
    scene_constants,
    scene_state,
    pb_renderer,
    tgt_width=1800
):
    """
    自动抽取夹爪 State 接近 0 和绝对值最大 的两帧，上下对比渲染
    """
    gripper_states = scene_constants['robot']['gripper_positions']

    # 🌟 1. 寻找两个极限帧的索引
    # 找绝对值最大的一帧 (通常是最大张开或最大抓取)
    idx_max = np.argmax(np.abs(gripper_states))
    val_max = gripper_states[idx_max]

    # 找最接近 0 的一帧 (通常是完全闭合或完全张开的另一个极限)
    idx_zero = np.argmin(np.abs(gripper_states))
    val_zero = gripper_states[idx_zero]

    print(f"🎯 锁定验证帧 [State ~ 0]: 第 {idx_zero} 帧 | 夹爪数据: {val_zero:.4f}")
    print(f"🎯 锁定验证帧 [State Max]: 第 {idx_max} 帧 | 夹爪数据: {val_max:.4f}")

    camera_ids = list(scene_constants['camera'].keys())
    wrist_serial = scene_constants['meta']['wrist_serial']

    # 🌟 2. 内部封装一个单帧渲染的闭包函数，避免代码重复
    def render_single_frame(frame_idx, gripper_val):
        current_joints = scene_constants['robot']['joint_positions'][frame_idx]
        pb_renderer.update_robot_pose(current_joints, gripper_state=gripper_val)

        frame_views = []
        for cam_id in camera_ids:
            cam_data = scene_constants['camera'][cam_id]
            cam_state = scene_state[cam_id]

            img_rgb = cam_data['video_rgb'][frame_idx].copy()
            h_img, w_img = img_rgb.shape[:2]

            # 渲染 Mask
            robot_mask = pb_renderer.render_mask(
                extrinsics=cam_state['extrinsics'][frame_idx],
                intrinsics=cam_data['K_mat'],
                width=w_img,
                height=h_img
            ) > 0

            # 赛博朋克风混色
            overlay = img_rgb.copy()
            overlay[robot_mask] = [50, 150, 255]
            blended_img = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

            # 文本与阴影绘制
            is_wrist = (cam_id == wrist_serial)
            cam_type = "Wrist Cam" if is_wrist else "Ext Cam"
            cv2.putText(blended_img, f"{cam_type} [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
            cv2.putText(blended_img, f"{cam_type} [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)

            frame_views.append(blended_img)

        # 横向拼接并缩放
        row_concat = np.concatenate(frame_views, axis=1)
        tgt_height = int(row_concat.shape[0] * (tgt_width / row_concat.shape[1]))
        return cv2.resize(row_concat, (tgt_width, tgt_height))

    # 🌟 3. 分别渲染两张极限帧的底片
    img_zero = render_single_frame(idx_zero, val_zero)
    img_max = render_single_frame(idx_max, val_max)

    # 🌟 4. 使用 matplotlib 上下分屏对比展示
    fig, axes = plt.subplots(2, 1, figsize=(18, 12))

    axes[0].imshow(img_zero)
    axes[0].set_title(f"Gripper State ~ 0 (Frame {idx_zero} | State {val_zero:.4f})", fontsize=16, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_max)
    axes[1].set_title(f"Gripper Maximum Value (Frame {idx_max} | State {val_max:.4f})", fontsize=16, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

# ================= 启动验证 =================
pb_renderer_ultimate = PyBulletRenderer_Robotiq()

inspect_gripper_extremes(
    scene_constants=scene_constants,
    scene_state=scene_state,
    pb_renderer=pb_renderer_ultimate,
    tgt_width=1800
)

In [ ]:
# @title 3D 单帧融合点云与交互式可视化

render_fused_point_cloud(
    scene_constants=scene_constants,
    scene_state=scene_state,
)

In [ ]:
# @title 4D 全域点云环绕视频

def get_look_at_matrix(eye, target, up=(0, 0, 1)):
    """3D 运镜辅助：一行流生成 OpenGL 风格相机朝向矩阵"""
    z_axis = np.array(eye, dtype=float) - target
    z_axis /= np.linalg.norm(z_axis) + 1e-6

    x_axis = np.cross(up, z_axis)
    x_axis /= np.linalg.norm(x_axis) + 1e-6

    y_axis = np.cross(z_axis, x_axis)

    view_matrix = np.eye(4)
    view_matrix[:3, :4] = np.column_stack((x_axis, y_axis, z_axis, eye))
    return view_matrix

def render_cinematic_4d_orbit(scene_constants, scene_state, max_render_points=400000, width=640,
                              height=360, orbit_center=(0.4, 0.0, 0.0), orbit_radius=1.2,
                              camera_height=0.5, angle_start=np.pi/2):

    # 🌟 1. 拥抱新架构：直接提取 camera 层级的 keys，彻底干掉丑陋的 if 过滤
    camera_ids = sorted(scene_constants['camera'].keys())
    n_frames = len(scene_state[camera_ids[0]]['extrinsics'])

    # 🌟 2. 场景与渲染器初始化
    scene = pyrender.Scene(bg_color=[0.0, 0.0, 0.0, 1.0])
    cam_node = scene.add(pyrender.PerspectiveCamera(yfov=np.pi/3.0, aspectRatio=width/height), pose=np.eye(4))
    light_node = scene.add(pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=4.0), pose=np.eye(4))
    renderer = pyrender.OffscreenRenderer(width, height)

    video_frames = []

    # 🌟 3. 极速 4D 渲染循环：加入语义化变量
    for frame_idx in tqdm(range(n_frames), desc=f"🎥 渲染 4D 运镜"):

        # --- A. 提取并融合多视角点云 ---
        points, colors = [], []
        for cam_id in camera_ids:
            cam_data = scene_constants['camera'][cam_id]
            cam_state = scene_state[cam_id]

            # 提取指定帧的 3D 空间点与色彩
            points_3d, colors_rgb = unproject_to_3d(
                cam_data['raw_depth'][frame_idx],
                cam_data['video_rgb'][frame_idx],
                cam_data['K_mat'],
                T_cam2world=cam_state['extrinsics'][frame_idx]
            )
            points.append(points_3d)
            colors.append(colors_rgb)

        points = np.vstack(points)
        colors = np.vstack(colors)

        # --- B. 显存保护机制 (降维打击，只需两行) ---
        sample_idx = np.random.permutation(len(points))[:max_render_points]
        points, colors = points[sample_idx], colors[sample_idx]

        # --- C. 计算平滑环绕运镜位姿 ---
        angle = angle_start + (frame_idx * np.pi / n_frames)
        eye_pos = [orbit_center[0] + orbit_radius * np.cos(angle),
                   orbit_center[1] + orbit_radius * np.sin(angle),
                   camera_height]

        viz_pose = get_look_at_matrix(eye_pos, orbit_center)
        scene.set_pose(cam_node, pose=viz_pose)
        scene.set_pose(light_node, pose=viz_pose)

        # --- D. 压入渲染、拔出销毁、盖水印一气呵成 ---
        mesh_node = scene.add(pyrender.Mesh.from_points(points, colors=colors))
        color_img, _ = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
        scene.remove_node(mesh_node)

        # 🌟 仅拷贝 RGB 通道打断只读锁定，跳过 Alpha 通道的冗余拷贝
        img_rgb = color_img[:, :, :3].copy()
        cv2.putText(img_rgb, f"Frame: {frame_idx:03d}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        video_frames.append(img_rgb)

    renderer.delete()
    return video_frames

# ================= 主流程 =================
output_frames = render_cinematic_4d_orbit(
    scene_constants=scene_constants,
    scene_state=scene_state,
)

media.show_video(output_frames, fps=15, codec='gif')

In [ ]:
# @title 剔除静止发呆帧 (双轨联动 + 自动推断切片版)

def filter_idle_frames(scene_constants, scene_state):
    """
    双轨联动：同时对 constants 和 state 进行安全深拷贝与全域静止帧切除。
    采用“自动长度侦测”魔法：凡是时间维度等于 n_original 的数据统统一刀切！
    """
    print("✂️ 启动全域发呆帧双轨切除程序...")

    # 🌟 1. 绝对防御：深拷贝，绝不污染你原本辛辛苦苦跑出来的数据大盘！
    filtered_constants = copy.deepcopy(scene_constants)
    filtered_state = copy.deepcopy(scene_state)

    valid_indices = filtered_constants['meta'].get('valid_indices')
    if valid_indices is None:
        print("  ⚠️ 未找到有效帧过滤数据，跳过切除。")
        return filtered_constants, filtered_state

    n_original = len(filtered_constants['robot']['joint_positions'])
    valid_indices = valid_indices[valid_indices < n_original]

    if len(valid_indices) == 0:
        print("  ❌ 警告：该视频过滤后没有任何有效帧保留！")
        return filtered_constants, filtered_state

    print(f"  ✅ 准备切除静止发呆帧！保留关键动作帧数: {len(valid_indices)} / {n_original}")

    # ==========================================
    # 🌟 2. 过滤 Constants (机器人本体 + 自动扫描相机数据)
    # ==========================================
    # 明确切除机器人本体流
    for key in ['joint_positions', 'gripper_positions', 'T_ee_base_all']:
        if key in filtered_constants['robot']:
            filtered_constants['robot'][key] = filtered_constants['robot'][key][valid_indices]

    # 相机多模态数据：动态侦测（无论你加了多少如 raw_depth, sam_masks，只要符合帧数长度，全部自动切除）
    for cam_id, cam_data in filtered_constants['camera'].items():
        for key, value in cam_data.items():
            if isinstance(value, (list, np.ndarray)) and len(value) == n_original:
                if isinstance(value, list):
                    cam_data[key] = [value[i] for i in valid_indices]
                else:
                    cam_data[key] = value[valid_indices]
                # print(f"    - 侦测到动态时序数据并切除: [{cam_id}] {key}")

    # ==========================================
    # 🌟 3. 过滤 Scene State (外部相机外参轨迹 + 腕部相机动态外参)
    # ==========================================
    if filtered_state is not None:
        for cam_id, state_data in filtered_state.items():
            # 找到动态重投影轨迹阵 (N, 4, 4)
            if 'extrinsics' in state_data and len(state_data['extrinsics']) == n_original:
                state_data['extrinsics'] = state_data['extrinsics'][valid_indices]
                # print(f"    - 侦测到动态相机姿态并切除: [{cam_id}] extrinsics")

    print("  🎉 双轨时间轴完全对齐！切除手术完美收官。")
    return filtered_constants, filtered_state

# ================= 启动调用与变量重命名 =================
# 命名规则：加上 active_ 前缀，代表这是“去除了静止发呆后的纯动作精炼版”
# @title 🔍 查验核对：全域切片后的数据结构大盘

# 1. 启动双轨切除与变量重命名
active_scene_constants, active_scene_state = filter_idle_frames(
    scene_constants=scene_constants,
    scene_state=scene_state
)

def lightweight_inspect(d, name="Dictionary", indent=0):
    """轻量级高颜值结构扫描器：专为查验时间轴 Shape 对齐而生"""
    if indent == 0:
        print(f"\n🧊 正在扫描数据大盘: 【{name}】")
        print("=" * 60)

    for key, value in d.items():
        # 跳过一些极其冗长且无时间维度的静态配置，让视野更聚焦
        if key in ['K_mat', 'D_mat', 'camera_models', 'meta']:
            continue

        spacing = "│   " * indent + "├── "

        if isinstance(value, dict):
            print(f"{'│   ' * indent}├── 📂 {key}/")
            lightweight_inspect(value, name, indent + 1)
        elif hasattr(value, 'shape'):  # 兼容 numpy 数组和 torch 张量
            print(f"{spacing}📊 {key:<20} -> shape: {value.shape}")
        elif isinstance(value, list):
            item_shape = value[0].shape if (len(value) > 0 and hasattr(value[0], 'shape')) else "mixed/scalar"
            print(f"{spacing}📜 {key:<20} -> list  (len={len(value)}), items: {item_shape}")
        else:
            if indent == 0:
                print(f"{spacing}📝 {key:<20} -> {type(value).__name__}")

# ================= 启动结构树打印 =================

if active_scene_constants:
    # 查验 Robot 本体流
    lightweight_inspect(active_scene_constants['robot'], name="active_scene_constants ['robot']")

    # 抽查一个相机的视觉流 (以 Wrist Cam 为例)
    wrist_cam = active_scene_constants['meta']['wrist_serial']
    lightweight_inspect(active_scene_constants['camera'][wrist_cam], name=f"active_scene_constants ['camera']['{wrist_cam}']")

if active_scene_state:
    # 查验外部 4x4 外参轨迹阵
    lightweight_inspect(active_scene_state, name="active_scene_state")

print("\n🎯 查验重点：请核对上面打印的所有 shape 的第 0 维（比如从 194 变成了 157），它们必须完全一致！")

In [ ]:
# @title 全域时序数据切片引擎 (支持末尾截取/随机截取)

def slice_scene_temporal_window(scene_constants, scene_state, n=48, mode='last'):
    """
    时序截断通用算子：一次性遍历并切片所有嵌套字典中的时序张量。
    - mode='last': 提取末尾 N 帧 (黄金收尾)
    - mode='random': 提取随机连续 N 帧
    """
    print(f"\n✂️ 启动时序截断算子：正在剥离全局 [{mode}] 的 {n} 帧...")

    # 🌟 1. 绝对防御：深拷贝，绝不污染原版数据
    new_constants = copy.deepcopy(scene_constants)
    new_state = copy.deepcopy(scene_state) if scene_state else None

    # 获取当前总帧数 (以 joint_positions 为准)
    current_frames = len(new_constants['robot']['joint_positions'])

    if current_frames <= n:
        print(f"  ✅ 当前总帧数 ({current_frames} 帧) 已满足要求，无需截断。")
        return new_constants, new_state

    # 🌟 2. 策略分发：计算切片索引
    if mode == 'last':
        start_idx = current_frames - n
        end_idx = current_frames
    elif mode == 'random':
        start_idx = random.randint(0, current_frames - n)
        end_idx = start_idx + n
    else:
        raise ValueError("mode 参数必须是 'last' 或 'random'")

    print(f"  🎯 原帧数: {current_frames} -> 目标帧数: {n} (截取区间: [{start_idx}:{end_idx}])")

    # 🌟 3. 内部切割闭包：一行流完成所有类型张量/列表的安全截取
    def slice_temporal_dict(d, target_len):
        for k, v in list(d.items()):
            if isinstance(v, (list, np.ndarray)) and len(v) == target_len:
                d[k] = v[start_idx:end_idx]

    # 执行切割：Robot 本体流
    slice_temporal_dict(new_constants['robot'], current_frames)

    # 执行切割：Camera 视觉流与物理状态
    for cam_id, cam_data in new_constants['camera'].items():
        slice_temporal_dict(cam_data, current_frames)
        # 洗掉之前跑过的旧追踪缓存，防患于未然
        cam_data.pop('tracks_2d', None)
        cam_data.pop('vis_2d', None)

        if new_state and cam_id in new_state:
            slice_temporal_dict(new_state[cam_id], current_frames)

    print(f"  ✅ 截断完毕！所有时序维度现已完美对齐。")
    return new_constants, new_state

# 末尾 48 帧
# sliced_scene_constants, sliced_scene_state = slice_scene_temporal_window(
#     active_scene_constants,
#     active_scene_state,
#     n=48,
#     mode='last'
# )

# 随机 48 帧
sliced_scene_constants, sliced_scene_state = slice_scene_temporal_window(
    active_scene_constants,
    active_scene_state,
    n=48,
    mode='random'
)

In [ ]:
# @title 2D 点追踪

def extract_2d_tracks(model, scene_constants):
    """利用 CoTracker3 对全视角视频进行 2D 稠密追踪"""
    print("  🎯 执行 CoTracker3 稠密追踪...")

    # 🌟 1. 结构升级：直接遍历全新的 camera 层级，彻底干掉冗余的 if 过滤
    for cam_id in scene_constants['camera']:
        cam_data = scene_constants['camera'][cam_id]

        # 🌟 2. 语义复苏：提取当前机位的 RGB 视频并进行张量维度转换
        video_tensor = torch.from_numpy(cam_data['video_rgb']).permute(0, 3, 1, 2)[None].float().to(device)

        # 🌟 3. 极速推理
        with torch.no_grad():
            pred_tracks, pred_vis = model(video_tensor, grid_size=30, grid_query_frame=0, backward_tracking=False)

        # 🌟 4. 优雅落盘：将追踪坐标与可见度掩码精准压入对应的相机口袋
        cam_data.update({
            'tracks_2d': pred_tracks[0].cpu().numpy(),
            'vis_2d': pred_vis[0].cpu().numpy()
        })

        # 🌟 4. 核心救命操作：手动斩断指针，强制清空当前机位残留的显存！
        del video_tensor
        del pred_tracks
        del pred_vis
        import gc
        gc.collect()
        torch.cuda.empty_cache()

    return scene_constants

def render_2d_tracking_video(video_frames, tracks, visibility, global_colors=None, linewidth=3, tracks_leave_trace=20):
    """极速版 2D 渲染引擎 (终极防线：连线也严格遵守遮挡掩码)"""
    n_frames, n_points, _ = tracks.shape
    point_radius = int(linewidth * 2)

    # 获取原始视频的真实分辨率
    h_img, w_img = video_frames[0].shape[:2]

    track_pts = tracks.copy()

    # 1. 基础物理边界屏蔽 (屏蔽飞出屏幕范围的幻觉坐标)
    is_valid = (track_pts[..., 0] >= 0) & (track_pts[..., 0] < w_img) & \
               (track_pts[..., 1] >= 0) & (track_pts[..., 1] < h_img)

    # 🌟 核心防鬼影连线修复：画拖尾不仅要求在屏幕内，还必须没被物理遮挡！
    is_drawable = is_valid & visibility

    # 2. 拷贝视频帧防污染，并取整坐标
    video_frames = [f.copy() for f in video_frames]
    track_pts = np.round(track_pts).astype(np.int32)

    # 3. 极简调色板
    if global_colors is None:
        y_coords = tracks[0, :, 1]
        norm = plt.Normalize(y_coords.min(), y_coords.max())
        global_colors = plt.cm.gist_rainbow(norm(y_coords))[:, :3] * 255
    point_colors = [tuple(map(int, c)) for c in global_colors]

    # 4. 影视级逐帧渲染循环
    for t in range(n_frames):
        current_img = video_frames[t]
        trace_len = min(t, tracks_leave_trace)

        # --- A. 绘制带有透明度渐变的流星拖尾 ---
        for step in range(trace_len):
            past_t = t - trace_len + step
            alpha = (step / (trace_len + 1)) ** 2
            overlay = current_img.copy()

            # 🌟 修复：使用 is_drawable 替代 is_valid！
            # 只有在过去和现在都实打实可见的点，才允许连线！瞬间斩断穿模的直线！
            valid_edges = np.where(is_drawable[past_t] & is_drawable[past_t + 1])[0]
            for i in valid_edges:
                cv2.line(overlay, tuple(track_pts[past_t, i]), tuple(track_pts[past_t + 1, i]), point_colors[i], linewidth, cv2.LINE_AA)
            cv2.addWeighted(overlay, alpha, current_img, 1 - alpha, 0, current_img)

        # --- B. 绘制当前跟踪点 (区分可见与被遮挡) ---
        occ_overlay = current_img.copy()
        has_occlusion = False

        # 依然只处理当前帧在画面内的点
        active_points = np.where(is_valid[t])[0]

        for i in active_points:
            pt_coord = tuple(track_pts[t, i])

            if visibility[t, i]:
                # 可见点：实心圆
                cv2.circle(current_img, pt_coord, point_radius, point_colors[i], -1, cv2.LINE_AA)
            else:
                # 遮挡点：空心圆圈，并标记渲染透明遮罩
                cv2.circle(occ_overlay, pt_coord, point_radius, point_colors[i], 1, cv2.LINE_AA)
                has_occlusion = True

        # 一次性混合当前帧的所有被遮挡点，避免反复调用 addWeighted
        if has_occlusion:
            cv2.addWeighted(occ_overlay, 0.35, current_img, 0.65, 0, current_img)

    return video_frames

# ================= 主流程 =================
from cotracker.predictor import CoTrackerPredictor

# 🌟 1. 模型加载
model = CoTrackerPredictor(checkpoint="/content/co-tracker/weights/cotracker3_offline.pth").to(device)

sliced_scene_constants = extract_2d_tracks(model, sliced_scene_constants)

# 🌟 3. 极速渲染验证
all_viz_videos = []

for cam_id in sliced_scene_constants['camera']:
    print(f"🎨 正在渲染相机 {cam_id} 的 2D 追踪特效 (最后 48 帧)...")
    cam_data = sliced_scene_constants['camera'][cam_id]

    frames = render_2d_tracking_video(
        cam_data['video_rgb'],
        cam_data['tracks_2d'],
        cam_data['vis_2d']
    )
    all_viz_videos.append(np.array(frames))

# 🌟 4. 播放最后 48 帧的追踪效果
media.show_video(np.concatenate(all_viz_videos, axis=2), fps=15, codec='gif', height=256)

In [ ]:
# @title 2D 点追踪重投影可视化

def lift_tracks_to_3d(tracks_2d, vis_2d, depth, K_mat, extrinsics):
    """模块 A: 纯粹的 3D 升维算子 (严谨越界阻断)"""
    n_frames, n_points = tracks_2d.shape[:2]
    h_img, w_img = depth.shape[1:3]
    traj_3d = np.zeros((n_frames, n_points, 3))
    zs_src = np.zeros((n_frames, n_points))

    for t in range(n_frames):
        pts = tracks_2d[t]

        # 🌟 严谨越界检查：飞出屏幕的点绝对不能生硬 clip 去查边缘深度，必须直接判死刑！
        in_bounds = (pts[:, 0] >= 0) & (pts[:, 0] < w_img) & (pts[:, 1] >= 0) & (pts[:, 1] < h_img)

        us = np.clip(np.round(pts[:, 0]).astype(int), 0, w_img - 1)
        vs = np.clip(np.round(pts[:, 1]).astype(int), 0, h_img - 1)
        z_raw = depth[t, vs, us].copy()

        # 🌟 核心防鬼影：被遮挡的、或者飞出屏幕的，强制抹除深度，斩断幻觉源头
        invalid_mask = (~vis_2d[t]) | (~in_bounds)
        z_raw[invalid_mask] = 0.0

        zs_src[t] = z_raw
        traj_3d[t] = unproject_points_np(pts[:, 0], pts[:, 1], zs_src[t], K_mat, extrinsics[t])

    return traj_3d, zs_src

def project_to_camera(traj_3d, zs_src, depth, K_mat, extrinsics, depth_margin=0.05):
    """模块 B: 目标视角解算算子 (清爽版掩码逻辑 + 严谨越界修复)"""
    n_frames, n_points = traj_3d.shape[:2]
    h_img, w_img = depth.shape[1:3]

    # 初始化：所有不合法坐标默认丢到宇宙边缘
    proj_trk = np.full((n_frames, n_points, 2), -1000.0)
    proj_vis = np.zeros((n_frames, n_points), dtype=bool)

    for t in range(n_frames):
        u, v, z_pred = project_points_np(traj_3d[t], K_mat, extrinsics[t])

        # 🌟 逻辑重构：先筛选出真正的“物理合法点”
        valid_z = (zs_src[t] > 0.05) & (z_pred > 0.05)
        in_bounds = (u >= 0) & (u < w_img) & (v >= 0) & (v < h_img)
        valid_mask = valid_z & in_bounds

        # 仅对合法点赋予正确的 2D 坐标 (其余保持 -1000.0)
        proj_trk[t, valid_mask, 0] = u[valid_mask]
        proj_trk[t, valid_mask, 1] = v[valid_mask]

        # 🌟 遮挡计算：彻底杜绝对 -1000.0 坐标查深度的丑陋逻辑
        if valid_mask.any():
            # 🌟 终极修复：增加 np.clip 限制！
            # 防止 1279.6 被 round 成 1280 导致越界崩溃
            ui = np.clip(np.round(u[valid_mask]).astype(int), 0, w_img - 1)
            vi = np.clip(np.round(v[valid_mask]).astype(int), 0, h_img - 1)

            depth_sensor = depth[t, vi, ui]
            is_occ = (depth_sensor > 0) & (depth_sensor < z_pred[valid_mask] - depth_margin)

            # 将未被遮挡的合法点标记为 True
            proj_vis[t, valid_mask] = ~is_occ

    return proj_trk, proj_vis

# ================= 数据解耦模块 =================
def compute_reprojection_data(src_cam, scene_constants, scene_state):
    """数据推导模块：处理 3D 轨迹与各视角的重投影结果"""
    camera_ids = list(scene_constants['camera'].keys())
    src_data = scene_constants['camera'][src_cam]
    src_state = scene_state[src_cam]
    tracks_2d = src_data['tracks_2d']
    vis_src = src_data['vis_2d']

    traj_3d, zs_src = lift_tracks_to_3d(
        tracks_2d=tracks_2d, vis_2d=vis_src, depth=src_data['raw_depth'], K_mat=src_data['K_mat'], extrinsics=src_state['extrinsics']
    )

    trk_dict, vis_dict = {}, {}
    for tgt_cam in camera_ids:
        # 🌟 源视角特权：底层数据直接阻断重投影！原汁原味返回！
        if tgt_cam == src_cam:
            trk_dict[tgt_cam] = tracks_2d
            vis_dict[tgt_cam] = vis_src
        else:
            tgt_data = scene_constants['camera'][tgt_cam]
            tgt_state = scene_state[tgt_cam]
            trk_dict[tgt_cam], vis_dict[tgt_cam] = project_to_camera(
                traj_3d=traj_3d, zs_src=zs_src, depth=tgt_data['raw_depth'], K_mat=tgt_data['K_mat'], extrinsics=tgt_state['extrinsics']
            )

    return traj_3d, zs_src, vis_src, trk_dict, vis_dict, tracks_2d

# ================= 渲染解耦模块 =================
def render_all_tracks(src_cam, tracks_2d, vis_src, trk_dict, vis_dict, scene_constants):
    """画笔模块：无脑接盘渲染"""
    camera_ids = list(scene_constants['camera'].keys())
    h_img, w_img = scene_constants['camera'][src_cam]['video_rgb'].shape[1:3]

    y_vals = tracks_2d[0, :, 1]
    colors = plt.cm.gist_rainbow(plt.Normalize(y_vals.min(), y_vals.max())(y_vals))[:, :3] * 255

    res_vids = []
    tgt_size = (320, int(320 * h_img / w_img))
    text_org = (20, 50)

    for tgt_cam in camera_ids:
        # 🌟 现在画笔不再需要知道谁是源视角了。
        # 源视角拿到的 vis_dict 已经是 vis_src；跨视角拿到的是 proj_vis。
        # 统统执行 & vis_src 即可实现完美连带拦截。
        combined_vis = vis_dict[tgt_cam] & vis_src

        frames = render_2d_tracking_video(
            video_frames=scene_constants['camera'][tgt_cam]['video_rgb'],
            tracks=trk_dict[tgt_cam],
            visibility=combined_vis,
            global_colors=colors,
            linewidth=4
        )

        label = f"Src:{src_cam} -> Tgt:{tgt_cam}"
        cam_vid = []
        for img in frames:
            cv2.putText(img, label, text_org, cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
            cv2.putText(img, label, text_org, cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
            cam_vid.append(cv2.resize(img, tgt_size))
        res_vids.append(np.array(cam_vid))

    return np.concatenate(res_vids, axis=2)

# ================= 主流程 =================
camera_grids = []

for cam_id in sliced_scene_constants['camera']:
    traj_3d, zs_src, vis_src, trk_dict, vis_dict, tracks_2d = compute_reprojection_data(
        cam_id, sliced_scene_constants, sliced_scene_state
    )

    grid_video = render_all_tracks(
        cam_id, tracks_2d, vis_src, trk_dict, vis_dict, sliced_scene_constants
    )
    camera_grids.append(grid_video)

if len(camera_grids) > 0:
    media.show_video(np.concatenate(camera_grids, axis=1), fps=15, codec='gif')
else:
    print("❌ 渲染失败，未能生成视频画面。")

In [ ]:
# @title 动态3D点可视化函数

# ==========================================
# 🌟 New: 4D Interactive Point Cloud Player Core Operator
# ==========================================
def show_animated_plotly_point_cloud(traj_3d, colors_rgb, title="Animated 3D Tracks", eye_pos=(0, -0.8, -1.5)):
    """Dynamic interactive 3D point cloud player, supports timeline dragging and playback"""
    T, N, _ = traj_3d.shape

    # 1. Convert colors to rgb strings supported by Plotly
    hex_colors = [f'rgb({int(r)},{int(g)},{int(b)})' for r, g, b in colors_rgb]

    # 2. Lock the global physical coordinate system boundaries to prevent wild shaking during playback
    x_min, x_max = np.nanmin(traj_3d[:, :, 0]), np.nanmax(traj_3d[:, :, 0])
    y_min, y_max = np.nanmin(traj_3d[:, :, 1]), np.nanmax(traj_3d[:, :, 1])
    z_min, z_max = np.nanmin(traj_3d[:, :, 2]), np.nanmax(traj_3d[:, :, 2])

    # 3. Build the base canvas for frame 0
    fig = go.Figure(
        data=[go.Scatter3d(
            x=traj_3d[0, :, 0], y=traj_3d[0, :, 1], z=traj_3d[0, :, 2],
            mode='markers', marker=dict(size=2.0, color=hex_colors)
        )]
    )

    # 4. Push in the skeleton data for all time steps
    frames = []
    for t in range(T):
        frames.append(go.Frame(
            data=[go.Scatter3d(x=traj_3d[t, :, 0], y=traj_3d[t, :, 1], z=traj_3d[t, :, 2])],
            name=str(t)
        ))
    fig.frames = frames

    # 5. Assemble the player panel and timeline
    sliders = [dict(
        steps=[dict(method='animate',
                    args=[[str(t)], dict(mode='immediate', frame=dict(duration=80, redraw=True), transition=dict(duration=0))],
                    label=f"F{t}") for t in range(T)],
        active=0, transition=dict(duration=0), x=0, y=0
    )]

    fig.update_layout(
        title=title,
        margin=dict(l=0, r=0, b=0, t=40),
        height=600, showlegend=False,
        scene=dict(
            aspectmode='data',
            xaxis=dict(range=[x_min, x_max], autorange=False),
            yaxis=dict(range=[y_min, y_max], autorange=False),
            zaxis=dict(range=[z_min, z_max], autorange=False),
            camera=dict(eye=dict(x=eye_pos[0], y=eye_pos[1], z=eye_pos[2]))
        ),
        updatemenus=[dict(
            type="buttons", showactive=False, x=0.05, y=1.1,
            buttons=[
                dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=80, redraw=True), transition=dict(duration=0), fromcurrent=True)]),
                dict(label="⏸ Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate", transition=dict(duration=0))])
            ]
        )],
        sliders=sliders
    )
    fig.show(renderer="colab")

### 3D 点追踪提取

In [ ]:
# @title 🦾 URDF 物理刚体轨迹生成器 (边缘排爆安全版)
import numpy as np
import cv2
import pybullet as p
from scipy.spatial.transform import Rotation as R
from tqdm import tqdm

class URDFKinematicsTracker:
    def __init__(self, pb_renderer):
        """专职利用 URDF 模型正向推演机械臂/夹爪轨迹的引擎"""
        self.pb_renderer = pb_renderer

    def get_link_transform(self, obj_id, link_id):
        """提取 PyBullet 中指定连杆的 4x4 变换矩阵"""
        if link_id == -1:
            pos, orn = p.getBasePositionAndOrientation(obj_id)
        else:
            state = p.getLinkState(obj_id, link_id)
            pos, orn = state[0], state[1]
        T = np.eye(4)
        T[:3, :3] = R.from_quat(orn).as_matrix()
        T[:3, 3] = pos
        return T

    def extract_robot_tracks(self, src_cam, scene_constants, scene_state, safe_margin=7):
        print(f"\n🦾 启动 URDF 纯物理运动学追踪 | 视角: [{src_cam}]")

        src_data = scene_constants['camera'][src_cam]
        src_state = scene_state[src_cam]
        K_mat = src_data['K_mat']
        extrinsics = src_state['extrinsics']
        h_img, w_img = src_data['video_rgb'][0].shape[:2]
        n_frames = len(src_data['video_rgb'])

        # 我们依然以 CoTracker 第 0 帧撒的点作为初始种子
        tracks_2d_t0 = src_data['tracks_2d'][0]

        # =========================================================
        # 1. 时光倒流：恢复第 0 帧状态，寻找落在机器人上的种子点
        # =========================================================
        joint_angles_t0 = scene_constants['robot']['joint_positions'][0]
        gripper_state_t0 = scene_constants['robot']['gripper_positions'][0]
        self.pb_renderer.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

        # 渲染第 0 帧的 Mask 和 Depth
        cam_pos = extrinsics[0][:3, 3]
        target_pos = cam_pos + extrinsics[0][:3, 2]
        view_matrix = p.computeViewMatrix(cam_pos, target_pos, -extrinsics[0][:3, 1])
        proj_matrix = self.pb_renderer._get_projection_matrix(K_mat, w_img, h_img)

        _, _, _, depth_buffer, seg_buffer = p.getCameraImage(
            w_img, h_img, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
            renderer=p.ER_BULLET_HARDWARE_OPENGL,
            flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX
        )

        # 解析 URDF 原生深度
        urdf_depth_t0 = 0.1 / (10.0 - 9.99 * np.reshape(depth_buffer, (h_img, w_img)))
        urdf_depth_t0 = np.where(urdf_depth_t0 < 9.9, urdf_depth_t0, 0.0)

        # 解析 Segmentation
        seg_array = np.reshape(seg_buffer, (h_img, w_img)).astype(np.int32)
        obj_ids = seg_array & 0xFFFFFF
        link_ids = (seg_array >> 24) - 1

        # 锁定落在本体或夹爪上的像素
        is_robot = (obj_ids == self.pb_renderer.robot_id) | (obj_ids == self.pb_renderer.ghost_id)

        # 🌟 核心升级：对机器人 Mask 进行腐蚀 (Erosion)，安全排爆边缘点！
        # 使用 safe_margin (默认 7x7) 的核向内收缩，确保追踪点深埋在机械臂内部
        kernel = np.ones((safe_margin, safe_margin), np.uint8)
        is_robot_safe = cv2.erode(is_robot.astype(np.uint8), kernel, iterations=1) > 0

        # 判断追踪点哪些属于机器人
        u0 = np.clip(np.round(tracks_2d_t0[:, 0]).astype(int), 0, w_img - 1)
        v0 = np.clip(np.round(tracks_2d_t0[:, 1]).astype(int), 0, h_img - 1)

        # 🌟 使用内缩后的绝对安全 Mask 进行筛选
        on_robot_mask = is_robot_safe[v0, u0]

        robot_indices = np.where(on_robot_mask)[0]

        if len(robot_indices) == 0:
            print("  ⚠️ 第 0 帧没有识别到任何安全的机械臂/夹爪点，无法执行物理追踪！")
            return None, None, None, None

        print(f"  🔍 成功在第 0 帧捕获 {len(robot_indices)} 个【安全内缩】的刚体表面点！")

        # 提取这些点对应的物体、连杆和深度
        robot_objs = obj_ids[v0[robot_indices], u0[robot_indices]]
        robot_links = link_ids[v0[robot_indices], u0[robot_indices]]
        z0 = urdf_depth_t0[v0[robot_indices], u0[robot_indices]]

        # 升维得到初始 3D 世界坐标
        pts_world_t0 = unproject_points_np(
            tracks_2d_t0[robot_indices, 0],
            tracks_2d_t0[robot_indices, 1],
            z0, K_mat, extrinsics[0]
        )

        # 将世界坐标绑定到局部连杆坐标系 (Local Frame)
        unique_parts = set(zip(robot_objs, robot_links))
        local_pts_dict = {}
        for obj_id, link_id in unique_parts:
            part_mask = (robot_objs == obj_id) & (robot_links == link_id)
            part_pts = pts_world_t0[part_mask]

            T_link_world_t0 = self.get_link_transform(obj_id, link_id)
            T_world_link_t0 = np.linalg.inv(T_link_world_t0)

            P_homo = np.hstack([part_pts, np.ones((len(part_pts), 1))]).T
            local_pts_dict[(obj_id, link_id)] = (part_mask, T_world_link_t0 @ P_homo)

        # =========================================================
        # 2. 正向推演：生成全序列 3D 与 2D 轨迹
        # =========================================================
        traj_3d = np.zeros((n_frames, len(robot_indices), 3), dtype=np.float32)
        traj_2d = np.zeros((n_frames, len(robot_indices), 2), dtype=np.float32)
        vis_2d = np.zeros((n_frames, len(robot_indices)), dtype=bool)

        for t in tqdm(range(n_frames), desc="  ⚙️ 正向运动学推演与遮挡计算"):
            # 更新物理引擎关节
            self.pb_renderer.update_robot_pose(
                scene_constants['robot']['joint_positions'][t],
                gripper_state=scene_constants['robot']['gripper_positions'][t]
            )

            # 根据连杆的位移，计算最新的 3D 世界坐标
            for obj_id, link_id in unique_parts:
                part_mask, P_local_homo = local_pts_dict[(obj_id, link_id)]
                T_link_world_t = self.get_link_transform(obj_id, link_id)
                P_world_t = T_link_world_t @ P_local_homo
                traj_3d[t, part_mask, :] = P_world_t[:3, :].T

            # 拍平到当前相机的 2D 平面
            u_t, v_t, z_pred_t = project_points_np(traj_3d[t], K_mat, extrinsics[t])
            traj_2d[t, :, 0] = u_t
            traj_2d[t, :, 1] = v_t

            # =========================================================
            # 3. 动态可见性判定 (越界 + 自遮挡 + 环境遮挡)
            # =========================================================
            urdf_depth_t = self.pb_renderer.render_depth(extrinsics[t], K_mat, w_img, h_img)
            raw_depth_t = src_data['raw_depth'][t]

            ui = np.clip(np.round(u_t).astype(int), 0, w_img - 1)
            vi = np.clip(np.round(v_t).astype(int), 0, h_img - 1)

            # 法则一：不能飞出屏幕
            in_bounds = (u_t >= 0) & (u_t < w_img) & (v_t >= 0) & (v_t < h_img) & (z_pred_t > 0)

            # 法则二：不能跑到机械臂自己背后 (2cm 容错)
            z_urdf = urdf_depth_t[vi, ui]
            not_self_occ = (z_urdf > 0) & (z_pred_t <= z_urdf + 0.015)

            # 法则三：不能被环境中其他物体遮挡 (3cm 容错)
            z_sensor = raw_depth_t[vi, ui]
            not_env_occ = ~((z_sensor > 0) & (z_pred_t > z_sensor + 0.02))

            vis_2d[t] = in_bounds & not_self_occ & not_env_occ

        return traj_3d, traj_2d, vis_2d, robot_indices

In [ ]:
# @title 👁️ URDF 物理 3D 轨迹的全域 2D 验证阵列
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
import mediapy as media

def project_and_check_visibility(traj_3d, tgt_cam, scene_constants, scene_state, pb_renderer):
    """将 3D 轨迹投影到目标相机，并执行严格的三重物理遮挡检测"""
    tgt_data = scene_constants['camera'][tgt_cam]
    tgt_state = scene_state[tgt_cam]
    K_mat = tgt_data['K_mat']
    extrinsics = tgt_state['extrinsics']
    n_frames = len(traj_3d)
    n_pts = traj_3d.shape[1]
    h_img, w_img = tgt_data['video_rgb'][0].shape[:2]

    tgt_traj_2d = np.zeros((n_frames, n_pts, 2), dtype=np.float32)
    tgt_vis_2d = np.zeros((n_frames, n_pts), dtype=bool)

    for t in range(n_frames):
        # 1. 强制同步物理引擎姿态，以便渲染深度图做遮挡判断
        current_joints = scene_constants['robot']['joint_positions'][t]
        current_gripper = scene_constants['robot']['gripper_positions'][t]
        pb_renderer.update_robot_pose(current_joints, gripper_state=current_gripper)

        # 2. 将 3D 轨迹拍平到目标相机的 2D 平面
        u_t, v_t, z_pred_t = project_points_np(traj_3d[t], K_mat, extrinsics[t])
        tgt_traj_2d[t, :, 0] = u_t
        tgt_traj_2d[t, :, 1] = v_t

        urdf_depth_t = pb_renderer.render_depth(extrinsics[t], K_mat, w_img, h_img)
        raw_depth_t = tgt_data['raw_depth'][t]

        ui = np.clip(np.round(u_t).astype(int), 0, w_img - 1)
        vi = np.clip(np.round(v_t).astype(int), 0, h_img - 1)

        # 3. 三重判定：视场越界 + 自遮挡 + 环境遮挡
        in_bounds = (u_t >= 0) & (u_t < w_img) & (v_t >= 0) & (v_t < h_img) & (z_pred_t > 0)
        z_urdf = urdf_depth_t[vi, ui]
        # 只要点比 URDF 表面深超过 1cm，说明它已经跑到机械臂内部或背面了
        not_self_occ = (z_urdf > 0) & (z_pred_t <= z_urdf + 0.01)
        z_sensor = raw_depth_t[vi, ui]
        # 只要点比真实世界的遮挡物深超过 1cm，说明它被桌子/其他物品挡住了
        not_env_occ = ~((z_sensor > 0) & (z_pred_t > z_sensor + 0.01))

        tgt_vis_2d[t] = in_bounds & not_self_occ & not_env_occ

    return tgt_traj_2d, tgt_vis_2d

def render_urdf_cross_view(src_cam, tgt_cam, scene_constants, scene_state, pb_renderer, traj_2d, vis_2d, robot_indices, y_vals_src):
    """渲染单一交叉视角的动画帧"""
    cam_data = scene_constants['camera'][tgt_cam]
    cam_state = scene_state[tgt_cam]
    video_frames = cam_data['video_rgb']
    K_mat = cam_data['K_mat']
    extrinsics = cam_state['extrinsics']
    n_frames = len(video_frames)
    h_img, w_img = video_frames[0].shape[:2]

    # 🌟 核心：使用 Source 视角的 Y 坐标初始化调色板，保证同一批点在不同视角下颜色绝对一致！
    norm = plt.Normalize(y_vals_src.min() - 1, y_vals_src.max() + 1)
    point_colors = plt.cm.gist_rainbow(norm(y_vals_src))[:, :3] * 255

    out_frames = []
    # 进度条描述精简一下，避免太长
    for t in tqdm(range(n_frames), desc=f"渲染 [Src: {src_cam[-4:]} -> Tgt: {tgt_cam[-4:]}]"):
        img = video_frames[t].copy()

        current_joints = scene_constants['robot']['joint_positions'][t]
        current_gripper = scene_constants['robot']['gripper_positions'][t]
        pb_renderer.update_robot_pose(current_joints, gripper_state=current_gripper)

        robot_mask = pb_renderer.render_mask(extrinsics[t], K_mat, w_img, h_img) > 0

        overlay = img.copy()
        overlay[robot_mask] = [50, 150, 255]
        img = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)

        for i in range(len(robot_indices)):
            if vis_2d[t, i]:
                pt = (int(np.round(traj_2d[t, i, 0])), int(np.round(traj_2d[t, i, 1])))
                color = tuple(map(int, point_colors[i]))
                cv2.circle(img, pt, 4, (0, 0, 0), -1, cv2.LINE_AA)
                cv2.circle(img, pt, 3, color, -1, cv2.LINE_AA)

        label = f"Src:{src_cam[-6:]} -> Tgt:{tgt_cam[-6:]}"
        cv2.putText(img, label, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
        cv2.putText(img, label, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # 缩小尺寸以适应九宫格展示，防止内存爆炸
        img = cv2.resize(img, (320, int(320 * h_img / w_img)))
        out_frames.append(img)

    return out_frames

# ================== 启动全域交叉验证 ==================
if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()

urdf_tracker = URDFKinematicsTracker(pb_renderer_ultimate)
camera_ids = list(sliced_scene_constants['camera'].keys())
all_grid_rows = []

print(f"\n🚀 启动全域交叉投影阵列构建...")

for src_cam in camera_ids:
    print(f"\n" + "="*60)
    print(f"🌍 正在以 [{src_cam}] 为源视角，提取 3D 物理轨迹...")

    # 1. 从 Source 视角提取原生 3D 物理轨迹
    traj_3d, src_traj_2d, src_vis_2d, robot_indices = urdf_tracker.extract_robot_tracks(
        src_cam, sliced_scene_constants, sliced_scene_state
    )

    if src_traj_2d is None:
        continue

    y_vals_src = src_traj_2d[0, :, 1]
    row_videos = []

    # 2. 将这批 3D 点投影到所有的 Target 视角（包括自己）
    for tgt_cam in camera_ids:
        if src_cam == tgt_cam:
            tgt_traj_2d, tgt_vis_2d = src_traj_2d, src_vis_2d
        else:
            print(f"  🎯 向目标视角 [{tgt_cam}] 投影并计算遮挡...")
            tgt_traj_2d, tgt_vis_2d = project_and_check_visibility(
                traj_3d, tgt_cam, sliced_scene_constants, sliced_scene_state, pb_renderer_ultimate
            )

        # 3. 渲染该交叉视角的视频
        vid_frames = render_urdf_cross_view(
            src_cam, tgt_cam, sliced_scene_constants, sliced_scene_state,
            pb_renderer_ultimate, tgt_traj_2d, tgt_vis_2d, robot_indices, y_vals_src
        )
        row_videos.append(np.array(vid_frames))

    # 将同一个 Source 下的多个 Target 横向拼接 (Width维度)
    all_grid_rows.append(np.concatenate(row_videos, axis=2))

# 4. 纵向拼接所有 Source 的行 (Height维度)
if len(all_grid_rows) > 0:
    print("\n🎬 渲染完毕！正在展示 3x3 URDF 交叉投影大盘：")
    combined_video = np.concatenate(all_grid_rows, axis=1)
    media.show_video(combined_video, fps=15, codec='gif', height=540)
else:
    print("\n❌ 提取失败。")

In [ ]:
# @title 提取纯净背景点

import numpy as np
import cv2
import mediapy as media
from scipy.ndimage import median_filter
from tqdm import tqdm


# =====================================================================
# 底层几何算子
# =====================================================================
def unproject_points_np(u, v, z, K, T_cam2world=None):
    """2D -> 3D 反投影"""
    x_cam = (u - K[0, 2]) * z / K[0, 0]
    y_cam = (v - K[1, 2]) * z / K[1, 1]
    pts_cam = np.stack([x_cam, y_cam, z, np.ones_like(z)], axis=0)
    if T_cam2world is None:
        return pts_cam[:3].T
    return (T_cam2world @ pts_cam)[:3].T


def project_points_np(pts_world, K, T_cam2world):
    """3D -> 2D 重投影"""
    T_w2c = np.linalg.inv(T_cam2world)
    pts_cam = T_w2c @ np.hstack([pts_world, np.ones((len(pts_world), 1))]).T
    z = pts_cam[2]
    u, v = np.zeros_like(z), np.zeros_like(z)
    ok = z > 0
    u[ok] = pts_cam[0, ok] / z[ok] * K[0, 0] + K[0, 2]
    v[ok] = pts_cam[1, ok] / z[ok] * K[1, 1] + K[1, 2]
    return u, v, z


# =====================================================================
# Zero-Tracking 提取引擎
# =====================================================================
def extract_universal_zerotrack(
    src_cam_id, scene_constants, scene_state, pb_renderer,
    target_cams=None, mode='dynamic_occlusion',
    grid_spacing=30, depth_margin=0.05, max_env_depth=5.0,
    residual_std_thresh=0.01,
):
    """
    统一 Zero-Tracking 引擎。

    Args:
      mode: 'strict_static' — 环境相机提取纯背景，附带腕部结界 +
            深度残差时序标准差过滤非静态物体。
            'dynamic_occlusion' — 腕部相机提取动态遮挡点。
      residual_std_thresh: strict_static 模式下，逐点深度残差的时序标准差阈值（m）。
            超过此值的点被判定为非静态（运动物体或传感器噪声边缘点），永久移除。
    """
    print(f"\n🌐 Zero-Tracking | 源: [{src_cam_id}] | 模式: {mode}")

    if target_cams is None:
        target_cams = list(scene_constants['camera'].keys())

    src_data = scene_constants['camera'][src_cam_id]
    src_state = scene_state[src_cam_id]
    wrist_serial = str(scene_constants['meta']['wrist_serial'])
    is_strict = (mode == 'strict_static')

    K_src = src_data['K_mat']
    ext_t0 = src_state['extrinsics'][0]
    depth_t0 = src_data['raw_depth'][0]
    T_frames, h_src, w_src = src_data['raw_depth'].shape

    # --- Phase 1: T=0 种子点提取 ---
    pb_renderer.update_robot_pose(
        scene_constants['robot']['joint_positions'][0],
        gripper_state=scene_constants['robot']['gripper_positions'][0],
    )
    robot_mask = pb_renderer.render_mask(ext_t0, K_src, w_src, h_src) > 0
    k_size = 15 if is_strict else 25
    robot_mask = cv2.dilate(robot_mask.astype(np.uint8),
                            np.ones((k_size, k_size), np.uint8)) > 0

    Y, X = np.mgrid[0:h_src:grid_spacing, 0:w_src:grid_spacing]
    u0, v0 = X.ravel(), Y.ravel()
    z0 = depth_t0[v0, u0]

    valid = ~robot_mask[v0, u0] & (z0 > 0.05) & (z0 < max_env_depth)
    u0, v0, z0 = u0[valid], v0[valid], z0[valid]
    if len(u0) == 0:
        print("  ⚠️ T=0 无合法种子点。")
        return None

    pts_3d = unproject_points_np(u0, v0, z0, K_src, ext_t0)
    N = len(pts_3d)
    print(f"  ✅ 提取 {N} 个 3D 种子点。")

    # --- Phase 2: 时空重投影 ---
    traj_2d = {c: np.zeros((T_frames, N, 2), np.float32) for c in target_cams}
    vis_2d = {c: np.zeros((T_frames, N), bool) for c in target_cams}
    is_bg = np.ones(N, bool)

    # 深度残差收集（仅 strict_static 使用）
    depth_residuals = np.full((T_frames, N), np.nan, dtype=np.float32)

    # wrist 结界参数（仅 strict_static 使用）
    if is_strict and wrist_serial in scene_constants['camera']:
        wrist_cam = scene_constants['camera'][wrist_serial]
        h_w, w_w = wrist_cam['raw_depth'][0].shape[:2]
        K_wrist = wrist_cam['K_mat']

    for t in tqdm(range(T_frames), desc="  ⚙️ 重投影"):
        pb_renderer.update_robot_pose(
            scene_constants['robot']['joint_positions'][t],
            gripper_state=scene_constants['robot']['gripper_positions'][t],
        )

        # strict_static: 腕部几何结界
        if is_strict and wrist_serial in scene_state:
            u_w, v_w, z_w = project_points_np(
                pts_3d, K_wrist, scene_state[wrist_serial]['extrinsics'][t])
            in_wrist = ((u_w >= 0) & (u_w < w_w) &
                        (v_w >= 0) & (v_w < h_w) & (z_w > 0) & (z_w < 1.5))
            is_bg[in_wrist] = False

        for cam in target_cams:
            cd = scene_constants['camera'][cam]
            K, ext = cd['K_mat'], scene_state[cam]['extrinsics'][t]
            h_c, w_c = cd['raw_depth'][t].shape[:2]

            u_t, v_t, z_pred = project_points_np(pts_3d, K, ext)
            traj_2d[cam][t, :, 0] = u_t
            traj_2d[cam][t, :, 1] = v_t

            in_bounds = ((u_t >= 0) & (u_t < w_c) &
                          (v_t >= 0) & (v_t < h_c) & (z_pred > 0))
            ui = np.clip(np.round(u_t).astype(int), 0, w_c - 1)
            vi = np.clip(np.round(v_t).astype(int), 0, h_c - 1)

            z_urdf = pb_renderer.render_depth(ext, K, w_c, h_c)[vi, ui]
            z_sens = cd['raw_depth'][t, vi, ui]

            occluded_self = (z_urdf > 0) & (z_pred > z_urdf + depth_margin)
            occluded_env = (z_sens > 0) & (z_pred > z_sens + depth_margin)
            visible = in_bounds & ~occluded_self & ~occluded_env

            if is_strict:
                # 收集深度残差（用源相机自身最稳定）
                if cam == src_cam_id:
                    testable = in_bounds & (z_sens > 0) & ~occluded_self
                    mask = testable & is_bg
                    depth_residuals[t, mask] = (z_sens - z_pred)[mask]

                # 传感器黑洞零阶保持（防闪烁）
                black = z_sens == 0
                fallback = vis_2d[cam][t - 1] if t > 0 else True
                vis_2d[cam][t] = np.where(black, fallback, visible)
            else:
                vis_2d[cam][t] = visible

    # --- Phase 2.5: 深度残差 std 一刀切（仅 strict_static）---
    if is_strict:
        residual_std = np.nanstd(depth_residuals, axis=0)
        residual_std = np.nan_to_num(residual_std, nan=999.0)

        bad = residual_std > residual_std_thresh
        n_killed = np.sum(is_bg & bad)
        is_bg[bad] = False
        print(f"  🗑️ 深度残差 std 淘汰 {n_killed} 个非静态点"
              f"（阈值: {residual_std_thresh*1000:.0f}mm）")

    # --- Phase 3: 组装输出 ---
    if is_strict:
        idx = np.where(is_bg)[0]
        if len(idx) == 0:
            return None
        pts_3d = pts_3d[idx]
        for cam in target_cams:
            traj_2d[cam] = traj_2d[cam][:, idx]
            vis_2d[cam] = median_filter(
                vis_2d[cam][:, idx].astype(float), size=(15, 1)) > 0.5
        print(f"  🏆 幸存纯背景点: {len(idx)}")

    traj_3d_out = np.tile(pts_3d, (T_frames, 1, 1))

    # 统一：不可见点坐标放逐到屏幕外
    for cam in target_cams:
        traj_2d[cam][~vis_2d[cam]] = -1000.0

    return {'traj_3d': traj_3d_out, 'traj_2d': traj_2d, 'vis_2d': vis_2d}


print("\n" + "=" * 50)
print("🚀 全视角 3D 种子点提取与合并...")
print("=" * 50)

wrist_serial = str(sliced_scene_constants['meta']['wrist_serial'])
camera_ids = list(sliced_scene_constants['camera'].keys())

all_traj_3d = []
all_traj_2d = {cam: [] for cam in camera_ids}
all_vis_2d = {cam: [] for cam in camera_ids}

# --- 1. 环境相机：严格静态点 ---
for cam_id in camera_ids:
    if str(cam_id) == wrist_serial:
        continue
    res = extract_universal_zerotrack(
        src_cam_id=cam_id, scene_constants=sliced_scene_constants,
        scene_state=sliced_scene_state, pb_renderer=pb_renderer_ultimate,
        target_cams=camera_ids, mode='strict_static',
        residual_std_thresh=0.01)
    if res:
        all_traj_3d.append(res['traj_3d'])
        for cam in camera_ids:
            all_traj_2d[cam].append(res['traj_2d'][cam])
            all_vis_2d[cam].append(res['vis_2d'][cam])

# --- 2. 腕部相机：动态遮挡点 ---
wrist_res = extract_universal_zerotrack(
    src_cam_id=wrist_serial, scene_constants=sliced_scene_constants,
    scene_state=sliced_scene_state, pb_renderer=pb_renderer_ultimate,
    target_cams=camera_ids, mode='dynamic_occlusion', grid_spacing=15)
if wrist_res:
    all_traj_3d.append(wrist_res['traj_3d'])
    for cam in camera_ids:
        all_traj_2d[cam].append(wrist_res['traj_2d'][cam])
        all_vis_2d[cam].append(wrist_res['vis_2d'][cam])

# --- 3. 合并 ---
if not all_traj_3d:
    raise ValueError("⚠️ 未能从任何视角获取合法点位。")

bg_traj_3d = np.concatenate(all_traj_3d, axis=1)
merged_traj_2d = {cam: np.concatenate(all_traj_2d[cam], axis=1) for cam in camera_ids}
merged_vis_2d = {cam: np.concatenate(all_vis_2d[cam], axis=1) for cam in camera_ids}
T_frames, N_total, _ = bg_traj_3d.shape
print(f"🏆 合并成功！共 {N_total} 个点。")

# --- 全局调色板 ---
y_vals = np.nanmean(bg_traj_3d[:, :, 1], axis=0)
y_valid = y_vals[~np.isnan(y_vals)]
norm = plt.Normalize(y_valid.min(), y_valid.max()) if len(y_valid) > 1 else plt.Normalize(-1, 1)
global_colors = (plt.cm.gist_rainbow(norm(y_vals))[:, :3] * 255).astype(np.uint8)

# --- 3D 可视化 ---
try:
    show_animated_plotly_point_cloud(
        traj_3d=bg_traj_3d, colors_rgb=global_colors,
        title="3D: Static Background", eye_pos=(0, -0.8, -1.5))
except NameError:
    print("  ⚠️ show_animated_plotly_point_cloud 未定义，跳过。")

# --- 2D 重投影可视化 ---
all_cam_videos = []
tgt_width = 400

for cam_id in camera_ids:
    cam_data = sliced_scene_constants['camera'][cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]
    tgt_height = int(tgt_width * h_img / w_img)

    rendered = render_2d_tracking_video(
        video_frames=cam_data['video_rgb'],
        tracks=merged_traj_2d[cam_id],
        visibility=merged_vis_2d[cam_id],
        global_colors=global_colors, linewidth=3)

    label = f"Cam: {cam_id} [WRIST]" if str(cam_id) == wrist_serial else f"Cam: {cam_id}"
    resized = []
    for img in rendered:
        cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 3)
        cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        resized.append(cv2.resize(img, (tgt_width, tgt_height)))
    all_cam_videos.append(np.array(resized))

if all_cam_videos:
    media.show_video(np.concatenate(all_cam_videos, axis=2), fps=15, codec='gif')

In [ ]:
# @title Phase 1: 每个视角独立 CoTracker Dense 2D Tracking

import numpy as np
import cv2
import torch
import mediapy as media
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm
from core.geometry import unproject_points_np, project_points_np


camera_ids = list(sliced_scene_constants['camera'].keys())
wrist_serial = str(sliced_scene_constants['meta']['wrist_serial'])

print("=" * 60)
print("🎯 Phase 1: 每个视角独立 CoTracker Dense 2D Tracking")
print("=" * 60)

for cam_id in camera_ids:
    cam_data = sliced_scene_constants['camera'][cam_id]

    # 如果已经跑过就跳过
    if 'tracks_2d' in cam_data and 'vis_2d' in cam_data:
        T, N, _ = cam_data['tracks_2d'].shape
        print(f"  ⏭️ [{cam_id}] 已有 {N} 条 2D tracks，跳过。")
        continue

    video_tensor = (
        torch.from_numpy(cam_data['video_rgb'])
        .permute(0, 3, 1, 2)[None].float().to(device)
    )
    with torch.no_grad():
        pred_tracks, pred_vis = model(
            video_tensor, grid_size=30, grid_query_frame=0,
            backward_tracking=False
        )
    cam_data['tracks_2d'] = pred_tracks[0].cpu().numpy()  # (T, N, 2)
    cam_data['vis_2d'] = pred_vis[0].cpu().numpy() > 0.5   # (T, N) bool
    T, N, _ = cam_data['tracks_2d'].shape
    print(f"  ✅ [{cam_id}] 提取 {N} 条 2D tracks，{T} 帧。")

print("\n✅ Phase 1 完成！")



In [ ]:
# @title Viz 1: 每个视角的原生 CoTracker 2D 轨迹

all_cam_videos = []
tgt_width = 400

for cam_id in camera_ids:
    cam_data = sliced_scene_constants['camera'][cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]
    tgt_height = int(tgt_width * h_img / w_img)

    tracks = cam_data['tracks_2d']
    vis = cam_data['vis_2d']
    T, N, _ = tracks.shape

    # 用 t=0 的 y 坐标上色
    y0 = tracks[0, :, 1]
    norm = plt.Normalize(y0.min(), y0.max())
    colors = (plt.cm.gist_rainbow(norm(y0))[:, :3] * 255).astype(np.uint8)

    rendered = render_2d_tracking_video(
        video_frames=cam_data['video_rgb'],
        tracks=tracks, visibility=vis,
        global_colors=colors, linewidth=3)

    label = f"[WRIST] {cam_id}" if str(cam_id) == wrist_serial else f"Cam: {cam_id}"
    resized = []
    for img in rendered:
        cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 3)
        cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        resized.append(cv2.resize(img, (tgt_width, tgt_height)))
    all_cam_videos.append(np.array(resized))

print(f"🎬 Phase 1 Viz: 各视角独立 CoTracker 2D 轨迹")
media.show_video(np.concatenate(all_cam_videos, axis=2), fps=15, codec='gif')



In [ ]:
# @title Phase 2: 第0帧 Lift to 3D + Robot Mask 过滤

print("=" * 60)
print("🦴 Phase 2: Lift to 3D + Robot Mask 过滤")
print("=" * 60)

# 存放每个视角的环境点信息
per_cam_env = {}

for cam_id in camera_ids:
    cam_data = sliced_scene_constants['camera'][cam_id]
    cam_state = sliced_scene_state[cam_id]
    tracks_2d = cam_data['tracks_2d']   # (T, N, 2)
    vis_2d = cam_data['vis_2d']         # (T, N)
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]
    T, N_total, _ = tracks_2d.shape

    # --- Robot mask at t=0 ---
    pb_renderer_ultimate.update_robot_pose(
        sliced_scene_constants['robot']['joint_positions'][0],
        gripper_state=sliced_scene_constants['robot']['gripper_positions'][0],
    )
    robot_mask = pb_renderer_ultimate.render_mask(
        cam_state['extrinsics'][0], cam_data['K_mat'], w_img, h_img
    ) > 0
    kernel = np.ones((15, 15), np.uint8)
    robot_mask_dilated = cv2.dilate(
        robot_mask.astype(np.uint8), kernel, iterations=1
    ) > 0

    # --- 查询每个 track 在 t=0 的位置 ---
    u0 = np.clip(np.round(tracks_2d[0, :, 0]).astype(int), 0, w_img - 1)
    v0 = np.clip(np.round(tracks_2d[0, :, 1]).astype(int), 0, h_img - 1)
    z0 = cam_data['raw_depth'][0, v0, u0]

    # 环境点 = 非机器人 + 有深度 + 深度合理
    is_env = ~robot_mask_dilated[v0, u0]
    has_depth = (z0 > 0.05) & (z0 < 5.0)
    env_mask = is_env & has_depth
    env_indices = np.where(env_mask)[0]

    if len(env_indices) == 0:
        print(f"  ⚠️ [{cam_id}] 无环境点，跳过。")
        per_cam_env[cam_id] = None
        continue

    # --- Lift to 3D ---
    pts_3d_t0 = unproject_points_np(
        tracks_2d[0, env_indices, 0],
        tracks_2d[0, env_indices, 1],
        z0[env_indices],
        cam_data['K_mat'],
        cam_state['extrinsics'][0]
    )

    per_cam_env[cam_id] = {
        'env_indices': env_indices,         # 在原始 tracks 中的索引
        'pts_3d_t0': pts_3d_t0,             # (N_env, 3) 世界坐标
        'tracks_2d': tracks_2d[:, env_indices, :],  # (T, N_env, 2)
        'vis_2d': vis_2d[:, env_indices],            # (T, N_env)
    }
    n_env = len(env_indices)
    n_robot = N_total - n_env - np.sum(~has_depth & is_env)
    print(f"  ✅ [{cam_id}] 环境点: {n_env} | 机器人点: {np.sum(~is_env)} | 无深度: {np.sum(~has_depth)}")

print("\n✅ Phase 2 完成！")



In [ ]:
# @title Viz 2: Robot Mask 叠加 + 环境点/机器人点分色显示

fig, axes = plt.subplots(1, len(camera_ids), figsize=(6 * len(camera_ids), 4))
if len(camera_ids) == 1:
    axes = [axes]

for idx, cam_id in enumerate(camera_ids):
    cam_data = sliced_scene_constants['camera'][cam_id]
    img = cam_data['video_rgb'][0].copy()
    h_img, w_img = img.shape[:2]

    if per_cam_env[cam_id] is not None:
        env_data = per_cam_env[cam_id]
        tracks_t0 = cam_data['tracks_2d'][0]

        # 画所有点：灰色 = 机器人/无效，绿色 = 环境
        for i in range(len(tracks_t0)):
            u, v = int(round(tracks_t0[i, 0])), int(round(tracks_t0[i, 1]))
            if 0 <= u < w_img and 0 <= v < h_img:
                if i in env_data['env_indices']:
                    cv2.circle(img, (u, v), 3, (0, 255, 0), -1)
                else:
                    cv2.circle(img, (u, v), 2, (128, 128, 128), -1)

    label = f"[WRIST] {cam_id}" if str(cam_id) == wrist_serial else f"Cam: {cam_id}"
    cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 3)
    cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    if per_cam_env[cam_id] is not None:
        n_env = len(per_cam_env[cam_id]['env_indices'])
        cv2.putText(img, f"Env: {n_env}", (15, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    axes[idx].imshow(img)
    axes[idx].set_title(label)
    axes[idx].axis('off')

plt.suptitle("Phase 2: Green=Environment | Gray=Robot/Invalid", fontsize=14)
plt.tight_layout()
plt.show()



In [ ]:
# @title Phase 3: 3D 最近邻匹配去重 → 统一点集

print("=" * 60)
print("🔗 Phase 3: 3D 最近邻匹配去重")
print("=" * 60)

MATCH_RADIUS = 0.015  # 1.5cm 以内视为同一个物理点

# --- 收集所有视角的 3D 种子 ---
all_pts = []
all_cam_labels = []   # (cam_id, local_index)
all_cam_indices = []  # 在 per_cam_env 里的 local index

offset = 0
cam_offsets = {}

for cam_id in camera_ids:
    if per_cam_env[cam_id] is None:
        continue
    pts = per_cam_env[cam_id]['pts_3d_t0']
    n = len(pts)
    all_pts.append(pts)
    cam_offsets[cam_id] = (offset, offset + n)
    for i in range(n):
        all_cam_labels.append((cam_id, i))
    offset += n

all_pts = np.concatenate(all_pts, axis=0)
N_all = len(all_pts)
print(f"  📊 总计 {N_all} 个 3D 种子点（来自 {len(cam_offsets)} 个视角）")

# --- Union-Find ---
parent = list(range(N_all))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

# --- KD-Tree 匹配 ---
tree = cKDTree(all_pts)
pairs = tree.query_pairs(r=MATCH_RADIUS)
print(f"  🔗 找到 {len(pairs)} 对近距离匹配")

for a, b in pairs:
    union(a, b)

# --- 构建统一点集 ---
from collections import defaultdict
groups = defaultdict(list)
for i in range(N_all):
    groups[find(i)].append(i)

N_unified = len(groups)
print(f"  🏆 去重后统一点集: {N_unified} 个唯一点（去掉 {N_all - N_unified} 个重复）")

# 建立 unified_id → per-cam 映射
unified_pts_3d = np.zeros((N_unified, 3), dtype=np.float32)
# unified_id → {cam_id: local_idx in per_cam_env}
unified_to_cam = [dict() for _ in range(N_unified)]

for uid, (root, members) in enumerate(groups.items()):
    # 3D 位置取中位数
    member_pts = all_pts[members]
    unified_pts_3d[uid] = np.median(member_pts, axis=0)

    for m in members:
        cam_id, local_idx = all_cam_labels[m]
        unified_to_cam[uid][cam_id] = local_idx

# 统计覆盖度
coverage_counts = np.array([len(d) for d in unified_to_cam])
for n_views in range(1, len(camera_ids) + 1):
    count = np.sum(coverage_counts >= n_views)
    print(f"    被 ≥{n_views} 个视角观测: {count} 点")

print("\n✅ Phase 3 完成！")



In [ ]:
# @title Viz 3: 匹配的点用相同颜色在各视角显示

# 为统一点集分配颜色
y_vals = unified_pts_3d[:, 1]
norm = plt.Normalize(y_vals.min(), y_vals.max()) if len(y_vals) > 1 else plt.Normalize(-1, 1)
unified_colors = (plt.cm.gist_rainbow(norm(y_vals))[:, :3] * 255).astype(np.uint8)

fig, axes = plt.subplots(1, len(camera_ids), figsize=(6 * len(camera_ids), 4))
if len(camera_ids) == 1:
    axes = [axes]

for idx, cam_id in enumerate(camera_ids):
    cam_data = sliced_scene_constants['camera'][cam_id]
    img = cam_data['video_rgb'][0].copy()

    # 画匹配到的统一点
    n_matched = 0
    for uid in range(N_unified):
        if cam_id not in unified_to_cam[uid]:
            continue
        local_idx = unified_to_cam[uid][cam_id]
        u = int(round(per_cam_env[cam_id]['tracks_2d'][0, local_idx, 0]))
        v = int(round(per_cam_env[cam_id]['tracks_2d'][0, local_idx, 1]))
        color = tuple(int(c) for c in unified_colors[uid])
        if 0 <= u < img.shape[1] and 0 <= v < img.shape[0]:
            cv2.circle(img, (u, v), 4, color, -1)
            n_matched += 1

    # 用白色圆环标出多视角匹配的点
    n_multi = 0
    for uid in range(N_unified):
        if cam_id not in unified_to_cam[uid] or len(unified_to_cam[uid]) < 2:
            continue
        local_idx = unified_to_cam[uid][cam_id]
        u = int(round(per_cam_env[cam_id]['tracks_2d'][0, local_idx, 0]))
        v = int(round(per_cam_env[cam_id]['tracks_2d'][0, local_idx, 1]))
        if 0 <= u < img.shape[1] and 0 <= v < img.shape[0]:
            cv2.circle(img, (u, v), 7, (255, 255, 255), 1)
            n_multi += 1

    label = f"[WRIST] {cam_id}" if str(cam_id) == wrist_serial else f"Cam: {cam_id}"
    cv2.putText(img, f"{label} | {n_matched} pts | {n_multi} shared",
                (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
    cv2.putText(img, f"{label} | {n_matched} pts | {n_multi} shared",
                (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    axes[idx].imshow(img)
    axes[idx].axis('off')

plt.suptitle("Phase 3: 统一颜色 = 同一3D点 | 白圈 = 多视角共享", fontsize=14)
plt.tight_layout()
plt.show()



In [ ]:
# @title Phase 4: CoTracker Query Mode 跨视角补全

print("=" * 60)
print("🔄 Phase 4: CoTracker Query Mode 跨视角补全")
print("=" * 60)

T_frames = len(sliced_scene_constants['camera'][camera_ids[0]]['video_rgb'])

# 输出结构: per_cam_tracks[cam_id] = (T, N_unified, 2), per_cam_vis[cam_id] = (T, N_unified)
per_cam_tracks = {cam: np.full((T_frames, N_unified, 2), -1000.0, np.float32) for cam in camera_ids}
per_cam_vis = {cam: np.zeros((T_frames, N_unified), bool) for cam in camera_ids}

# Step 1: 填充已有的原生 track（来自独立 CoTracker 的结果）
for uid in range(N_unified):
    for cam_id, local_idx in unified_to_cam[uid].items():
        per_cam_tracks[cam_id][:, uid, :] = per_cam_env[cam_id]['tracks_2d'][:, local_idx, :]
        per_cam_vis[cam_id][:, uid] = per_cam_env[cam_id]['vis_2d'][:, local_idx]

# 统计需要补全的点
for cam_id in camera_ids:
    n_have = np.sum(per_cam_vis[cam_id][0])
    n_need = N_unified - n_have
    print(f"  [{cam_id}] 已有: {n_have} | 需补全: {n_need}")

# Step 2: 对每个视角，找出缺失的点，用 3D→2D 做 seed 跑 CoTracker query mode
for cam_id in camera_ids:
    cam_data = sliced_scene_constants['camera'][cam_id]
    cam_state = sliced_scene_state[cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]

    # 找到该视角缺失的 unified points
    missing_uids = []
    for uid in range(N_unified):
        if cam_id not in unified_to_cam[uid]:
            missing_uids.append(uid)

    if len(missing_uids) == 0:
        print(f"  ⏭️ [{cam_id}] 无缺失，跳过。")
        continue

    # 3D→2D 粗投影到 t=0 作为 seed
    pts_3d_missing = unified_pts_3d[missing_uids]
    u_seed, v_seed, z_seed = project_points_np(
        pts_3d_missing, cam_data['K_mat'], cam_state['extrinsics'][0]
    )

    # 过滤：在画面内 + 深度合理 + 没被遮挡
    ui_seed = np.clip(np.round(u_seed).astype(int), 0, w_img - 1)
    vi_seed = np.clip(np.round(v_seed).astype(int), 0, h_img - 1)
    z_sensor = cam_data['raw_depth'][0, vi_seed, ui_seed]

    in_bounds = (u_seed >= 0) & (u_seed < w_img) & (v_seed >= 0) & (v_seed < h_img) & (z_seed > 0)
    not_occluded = (z_sensor > 0) & (z_sensor >= z_seed - 0.02)
    valid_seed = in_bounds & not_occluded

    valid_missing_uids = [missing_uids[i] for i in range(len(missing_uids)) if valid_seed[i]]
    valid_u = u_seed[valid_seed]
    valid_v = v_seed[valid_seed]

    if len(valid_missing_uids) == 0:
        print(f"  ⚠️ [{cam_id}] 所有缺失点在此视角不可见，跳过。")
        continue

    print(f"  🎯 [{cam_id}] 对 {len(valid_missing_uids)} 个缺失点跑 CoTracker query mode...")

    # 构造 queries: (1, M, 3) → [t, x, y]
    queries_np = np.stack([
        np.zeros(len(valid_u)),
        valid_u,
        valid_v
    ], axis=-1)  # (M, 3)
    queries_t = torch.tensor(queries_np, dtype=torch.float32, device=device)[None]

    video_tensor = (
        torch.from_numpy(cam_data['video_rgb'])
        .permute(0, 3, 1, 2)[None].float().to(device)
    )

    with torch.no_grad():
        pred_tracks, pred_vis = model(
            video_tensor, queries=queries_t, backward_tracking=True
        )

    new_tracks = pred_tracks[0].cpu().numpy()   # (T, M, 2)
    new_vis = pred_vis[0].cpu().numpy() > 0.5    # (T, M)

    # 填入结果
    for j, uid in enumerate(valid_missing_uids):
        per_cam_tracks[cam_id][:, uid, :] = new_tracks[:, j, :]
        per_cam_vis[cam_id][:, uid] = new_vis[:, j]

# 最终统计
print("\n📊 补全后覆盖率:")
for cam_id in camera_ids:
    n_visible_t0 = np.sum(per_cam_vis[cam_id][0])
    pct = n_visible_t0 / N_unified * 100
    print(f"  [{cam_id}] t=0 可见: {n_visible_t0}/{N_unified} ({pct:.1f}%)")

print("\n✅ Phase 4 完成！")



In [ ]:
# @title Viz 4: 补全后各视角 2D 轨迹动画（原生=实心 | 补全=空心）

all_cam_videos = []
tgt_width = 400

for cam_id in camera_ids:
    cam_data = sliced_scene_constants['camera'][cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]
    tgt_height = int(tgt_width * h_img / w_img)

    rendered = render_2d_tracking_video(
        video_frames=cam_data['video_rgb'],
        tracks=per_cam_tracks[cam_id],
        visibility=per_cam_vis[cam_id],
        global_colors=unified_colors, linewidth=3)

    # 标注原生 vs 补全
    n_native = sum(1 for uid in range(N_unified) if cam_id in unified_to_cam[uid])
    n_filled = np.sum(per_cam_vis[cam_id][0]) - n_native

    label1 = f"[WRIST] {cam_id}" if str(cam_id) == wrist_serial else f"Cam: {cam_id}"
    label2 = f"Native:{n_native} | Filled:{max(0,int(n_filled))}"

    resized = []
    for img in rendered:
        cv2.putText(img, label1, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
        cv2.putText(img, label1, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        cv2.putText(img, label2, (15, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
        cv2.putText(img, label2, (15, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        resized.append(cv2.resize(img, (tgt_width, tgt_height)))
    all_cam_videos.append(np.array(resized))

print("🎬 Phase 4 Viz: 补全后跨视角统一 2D 轨迹")
media.show_video(np.concatenate(all_cam_videos, axis=2), fps=15, codec='gif')



In [ ]:
# @title Phase 5: 多视角多帧 Median 3D 融合

print("=" * 60)
print("📐 Phase 5: 多视角多帧 Median 3D 融合")
print("=" * 60)

# 每帧每视角每点 lift to 3D，然后 nanmedian 融合
fused_traj_3d = np.full((T_frames, N_unified, 3), np.nan, dtype=np.float32)

# 收集所有视角的 3D 观测
per_view_3d = np.full((len(camera_ids), T_frames, N_unified, 3), np.nan, dtype=np.float32)

for v_idx, cam_id in enumerate(camera_ids):
    cam_data = sliced_scene_constants['camera'][cam_id]
    cam_state = sliced_scene_state[cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]

    for t in range(T_frames):
        tracks_t = per_cam_tracks[cam_id][t]   # (N_unified, 2)
        vis_t = per_cam_vis[cam_id][t]          # (N_unified,)

        u = tracks_t[:, 0]
        v = tracks_t[:, 1]
        ui = np.clip(np.round(u).astype(int), 0, w_img - 1)
        vi = np.clip(np.round(v).astype(int), 0, h_img - 1)
        z = cam_data['raw_depth'][t, vi, ui]

        valid = vis_t & (z > 0.05) & (z < 5.0) & (u >= 0) & (u < w_img) & (v >= 0) & (v < h_img)

        if valid.any():
            pts_3d = unproject_points_np(
                u[valid], v[valid], z[valid],
                cam_data['K_mat'], cam_state['extrinsics'][t]
            )
            per_view_3d[v_idx, t, valid, :] = pts_3d

# Median 融合
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    fused_traj_3d = np.nanmedian(per_view_3d, axis=0)  # (T, N_unified, 3)

# 统计每个点被多少视角观测到
observation_counts = np.sum(~np.isnan(per_view_3d[:, :, :, 0]), axis=0)  # (T, N_unified)
avg_obs = np.nanmean(observation_counts, axis=0)  # (N_unified,)
print(f"  📊 平均每点每帧被 {np.nanmean(avg_obs):.1f} 个视角观测到")

# Gaussian 平滑
import pandas as pd
smoothed_3d = fused_traj_3d.copy()
valid_any = ~np.all(np.isnan(fused_traj_3d[:, :, 0]), axis=0)  # 至少有一帧有效
n_valid = np.sum(valid_any)
print(f"  📊 有有效观测的点: {n_valid}/{N_unified}")

if n_valid > 0:
    df = pd.DataFrame(smoothed_3d[:, valid_any, :].reshape(T_frames, -1))
    df = df.interpolate(method='linear', limit_direction='both')
    interpolated = df.to_numpy().reshape(T_frames, n_valid, 3)
    smoothed_valid = gaussian_filter1d(interpolated, sigma=1.5, axis=0)

    smoothed_3d[:, valid_any, :] = smoothed_valid

# 最终 visibility: 至少 2 个视角在该帧有观测
min_views_for_vis = 2
fused_vis = observation_counts >= min_views_for_vis  # (T, N_unified)
print(f"  📊 用 ≥{min_views_for_vis} 视角阈值: 平均每帧 {np.mean(np.sum(fused_vis, axis=1)):.0f} 个可见点")

# 质量过滤: 去掉总可见帧数太少的点
min_visible_frames = 5
total_visible = np.sum(fused_vis, axis=0)
quality_mask = total_visible >= min_visible_frames
n_survived = np.sum(quality_mask)
print(f"  🏆 质量过滤 (≥{min_visible_frames} 帧可见): {n_survived}/{N_unified} 点存活")

# 过滤
final_traj_3d = smoothed_3d[:, quality_mask, :]
final_vis_global = fused_vis[:, quality_mask]
final_per_cam_tracks = {cam: per_cam_tracks[cam][:, quality_mask, :] for cam in camera_ids}
final_per_cam_vis = {cam: per_cam_vis[cam][:, quality_mask] for cam in camera_ids}
final_colors = unified_colors[quality_mask]
N_final = n_survived

print(f"\n✅ Phase 5 完成！最终 {N_final} 个 3D 轨迹。")



In [ ]:
# @title Viz 5a: 融合后 3D 点云动画

try:
    show_animated_plotly_point_cloud(
        traj_3d=final_traj_3d,
        colors_rgb=final_colors,
        title=f"Fused 3D: {N_final} points",
        eye_pos=(0, -0.8, -1.5))
except NameError:
    print("  ⚠️ show_animated_plotly_point_cloud 未定义，跳过。")



In [ ]:
# @title Viz 5b: 每个视角的原生2D轨迹 + 3D→2D重投影误差比对

all_cam_videos = []
tgt_width = 400

for cam_id in camera_ids:
    cam_data = sliced_scene_constants['camera'][cam_id]
    cam_state = sliced_scene_state[cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]
    tgt_height = int(tgt_width * h_img / w_img)

    frames = []
    for t in range(T_frames):
        img = cam_data['video_rgb'][t].copy()

        for uid in range(N_final):
            if not final_per_cam_vis[cam_id][t, uid]:
                continue

            # 原生 CoTracker 2D (绿色)
            u_native = final_per_cam_tracks[cam_id][t, uid, 0]
            v_native = final_per_cam_tracks[cam_id][t, uid, 1]
            p_native = (int(round(u_native)), int(round(v_native)))

            # 3D→2D 重投影 (红色)
            pt_3d = final_traj_3d[t, uid]
            if np.isnan(pt_3d[0]):
                continue
            u_proj, v_proj, z_proj = project_points_np(
                pt_3d[None], cam_data['K_mat'], cam_state['extrinsics'][t]
            )
            p_proj = (int(round(u_proj[0])), int(round(v_proj[0])))

            color = tuple(int(c) for c in final_colors[uid])
            # 绿色实心 = 原生 CoTracker
            cv2.circle(img, p_native, 3, color, -1)
            # 红色圆环 = 3D→2D 重投影
            if z_proj[0] > 0:
                cv2.circle(img, p_proj, 5, (255, 0, 0), 1)
                # 连线显示误差
                cv2.line(img, p_native, p_proj, (255, 0, 0), 1)

        label = f"[WRIST] {cam_id}" if str(cam_id) == wrist_serial else f"Cam: {cam_id}"
        cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
        cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        cv2.putText(img, "Dot=CoTracker | Ring=3D Reproj", (15, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

        frames.append(cv2.resize(img, (tgt_width, tgt_height)))

    all_cam_videos.append(np.array(frames))

print("🎬 Viz 5b: 彩点=原生CoTracker | 红圈=3D融合重投影 | 红线=误差")
media.show_video(np.concatenate(all_cam_videos, axis=2), fps=15, codec='gif')



In [ ]:
# @title Phase 6: 最终输出统计 + 综合 2D 轨迹可视化

print("=" * 60)
print("📦 Phase 6: 最终输出")
print("=" * 60)
print(f"  final_traj_3d:             shape = {final_traj_3d.shape}")
for cam_id in camera_ids:
    print(f"  final_per_cam_tracks[{cam_id}]: shape = {final_per_cam_tracks[cam_id].shape}")
    print(f"  final_per_cam_vis[{cam_id}]:    shape = {final_per_cam_vis[cam_id].shape}")

# 不可见点坐标放逐到屏幕外
export_traj_2d = {}
export_vis_2d = {}
for cam_id in camera_ids:
    traj = final_per_cam_tracks[cam_id].copy()
    vis = final_per_cam_vis[cam_id].copy()
    traj[~vis] = -1000.0
    export_traj_2d[cam_id] = traj
    export_vis_2d[cam_id] = vis

print(f"\n🏆 最终产出:")
print(f"  • {N_final} 个统一 3D 轨迹")
print(f"  • {len(camera_ids)} 个视角的原生 2D 轨迹 + visibility")
print(f"  • 平均每帧可见点: {np.mean(np.sum(final_vis_global, axis=1)):.0f}")



In [ ]:
# @title Viz 6: 最终综合多视角 2D 轨迹动画

all_cam_videos = []
tgt_width = 400

for cam_id in camera_ids:
    cam_data = sliced_scene_constants['camera'][cam_id]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]
    tgt_height = int(tgt_width * h_img / w_img)

    rendered = render_2d_tracking_video(
        video_frames=cam_data['video_rgb'],
        tracks=export_traj_2d[cam_id],
        visibility=export_vis_2d[cam_id],
        global_colors=final_colors, linewidth=3)

    label = f"[WRIST] {cam_id}" if str(cam_id) == wrist_serial else f"Cam: {cam_id}"
    n_vis = np.sum(export_vis_2d[cam_id][0])
    resized = []
    for img in rendered:
        cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 3)
        cv2.putText(img, label, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        cv2.putText(img, f"Vis@t0: {n_vis}/{N_final}", (15, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        resized.append(cv2.resize(img, (tgt_width, tgt_height)))
    all_cam_videos.append(np.array(resized))

print(f"🎬 Final: 统一 {N_final} 点 × {len(camera_ids)} 视角 原生 CoTracker 2D 轨迹")
media.show_video(np.concatenate(all_cam_videos, axis=2), fps=15, codec='gif')



In [ ]:
# @title 👁️ 终极 2D 追踪多视角全景投影阵列 (双轨大一统版)
import numpy as np
import cv2
import matplotlib.pyplot as plt
import mediapy as media
from tqdm import tqdm

print("🔄 正在提取双轨大一统真值，并渲染多视角 2D 追踪流星阵列...")

# 确保追踪引擎就绪
if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()
if 'urdf_tracker' not in locals():
    urdf_tracker = URDFKinematicsTracker(pb_renderer_ultimate)
if 'consensus_tracker' not in locals():
    consensus_tracker = ConsensusVisualTracker(cotracker_model, device)

camera_ids = list(sliced_scene_constants['camera'].keys())
all_grid_rows = []

for src_cam in camera_ids:
    print(f"\n" + "="*60)
    print(f"🎬 正在以 [{src_cam}] 为源视角提取双轨白金真值...")

    cam_data = sliced_scene_constants['camera'][src_cam]
    cam_state = sliced_scene_state[src_cam]
    h_img, w_img = cam_data['video_rgb'][0].shape[:2]
    T_frames = len(cam_data['video_rgb'])
    tracks_2d = cam_data['tracks_2d']
    raw_depth = cam_data['raw_depth']

    orig_2d_all = []
    traj_3d_all = []
    vis_all = []

    # =========================================================
    # 🦾 Track B: 提取 URDF 机械臂纯物理点
    # =========================================================
    traj_3d_rob, proj_2d_rob, vis_rob, robot_indices = urdf_tracker.extract_robot_tracks(
        src_cam, sliced_scene_constants, sliced_scene_state
    )
    if proj_2d_rob is not None and len(robot_indices) > 0:
        orig_2d_all.append(tracks_2d[:, robot_indices, :])
        traj_3d_all.append(traj_3d_rob)
        vis_all.append(vis_rob)

    # =========================================================
    # 👁️ Track A: 提取纯环境点并利用核心算子终极提纯
    # =========================================================
    track_A_res = extract_and_filter_track_A(
        src_cam, sliced_scene_constants, sliced_scene_state,
        pb_renderer_ultimate, consensus_tracker
    )

    if track_A_res is not None:
        orig_2d_all.append(track_A_res['orig_2d'])
        traj_3d_all.append(track_A_res['traj_3d'])
        vis_all.append(track_A_res['vis_2d'])

    # =========================================================
    # 🤝 双轨合并与渲染大盘构建
    # =========================================================
    if not orig_2d_all:
        print(f"  ⚠️ 视角 [{src_cam}] 无存活点，跳过。")
        continue

    combined_orig_2d = np.concatenate(orig_2d_all, axis=1)
    combined_traj_3d = np.concatenate(traj_3d_all, axis=1)
    combined_vis = np.concatenate(vis_all, axis=1)
    N_pts = combined_traj_3d.shape[1]

    print(f"  🏆 提取成功！共 {N_pts} 个双轨极品点。开始投影至全域视角...")

    y_coords = combined_orig_2d[0, :, 1]
    if len(y_coords) > 1 and y_coords.max() > y_coords.min():
        norm = plt.Normalize(y_coords.min(), y_coords.max())
    else:
        norm = plt.Normalize(y_coords.min() - 1, y_coords.min() + 1)
    point_colors = plt.cm.gist_rainbow(norm(y_coords))[:, :3] * 255

    row_videos = []

    for tgt_cam in camera_ids:
        tgt_data = sliced_scene_constants['camera'][tgt_cam]
        tgt_state = sliced_scene_state[tgt_cam]
        tgt_video = tgt_data['video_rgb']

        # 🌟 绝杀 Bug：用 -1000.0 替代 0.0 初始化，防止渲染器在 [0,0] 处狂画幽灵圆圈！
        proj_2d = np.full((T_frames, N_pts, 2), -1000.0, dtype=np.float32)
        proj_vis = np.zeros((T_frames, N_pts), dtype=bool)

        for t in range(T_frames):
            valid_3d = ~np.isnan(combined_traj_3d[t, :, 2])
            if valid_3d.any():
                u, v, z_p = project_points_np(combined_traj_3d[t, valid_3d], tgt_data['K_mat'], tgt_state['extrinsics'][t])

                ui = np.clip(np.round(u).astype(int), 0, w_img - 1)
                vi = np.clip(np.round(v).astype(int), 0, h_img - 1)

                in_bounds = (u >= 0) & (u < w_img) & (v >= 0) & (v < h_img) & (z_p > 0)
                z_sensor = tgt_data['raw_depth'][t, vi, ui]
                xray_occluded = (z_sensor > 0) & (z_sensor < z_p - 0.05)

                vis_t = np.zeros(N_pts, dtype=bool)
                vis_t[valid_3d] = in_bounds & (~xray_occluded)

                proj_vis[t] = vis_t & combined_vis[t]

                # 🌟 绝杀 Bug：用 -1000.0 初始化单帧坐标池！
                coords = np.full((N_pts, 2), -1000.0, dtype=np.float32)
                coords[valid_3d, 0] = u
                coords[valid_3d, 1] = v
                proj_2d[t] = coords

        print(f"    🎨 渲染流星特效 [Src: {src_cam[-4:]} -> Tgt: {tgt_cam[-4:]}]...")
        rendered_frames = render_2d_tracking_video(
            video_frames=tgt_video,
            tracks=proj_2d,
            visibility=proj_vis,
            global_colors=point_colors,
            linewidth=3
        )

        tgt_h, tgt_w = 256, int(256 * w_img / h_img)
        cam_vid = []
        for img in rendered_frames:
            label = f"Src:{src_cam[-6:]} -> Tgt:{tgt_cam[-6:]}"
            cv2.putText(img, label, (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 4)
            cv2.putText(img, label, (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
            cam_vid.append(cv2.resize(img, (tgt_w, tgt_h)))

        row_videos.append(np.array(cam_vid))

    all_grid_rows.append(np.concatenate(row_videos, axis=2))

if len(all_grid_rows) > 0:
    print("\n🎬 渲染完毕！正在展示 3x3 双轨大一统 2D 追踪流星阵列：")
    combined_video = np.concatenate(all_grid_rows, axis=1)
    media.show_video(combined_video, fps=15, codec='gif')
else:
    print("\n❌ 没有任何可展示的追踪阵列。")

In [ ]:
# @title 4D 彩虹流星可视化 (双轨大一统真值接入安全防爆版 - Polyscope 霓虹管版)
import polyscope as ps
import mediapy as media
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# =========================================================
# 渲染核心模块：完全解耦，直接接收清洗好的 3D 轨迹
# =========================================================
def render_4d_rainbow_meteor_flow(scene_constants, scene_state, src_cam,
                                 traj_3d, traj_valid, point_colors,
                                 min_depth=0.15, max_depth=1.5, max_env_points=300000,
                                 tail_length=100, width=640, height=360, pullback_dist=0.85):
    # 1. 自动环境解析
    camera_ids = sorted(scene_constants['camera'].keys())
    wrist_serial = scene_constants['meta']['wrist_serial']
    ext_cams = [cam for cam in camera_ids if cam != wrist_serial]
    cam_start, cam_end = ext_cams[0], ext_cams[-1]

    n_frames = len(scene_constants['camera'][cam_start]['video_rgb'])

    if traj_3d.shape[1] == 0:
        print(f"⚠️ 视角 [{src_cam}] 无存活的极品点，无法渲染流星！")
        return []

    # 🌟 2. Polyscope 影视级初始化
    ps.set_allow_headless_backends(True)
    ps.init()
    ps.set_up_dir("z_up")

    # 【极夜模式】深邃的暗调背景，让霓虹灯彻底爆发
    ps.set_background_color((0., 0., 0.))
    ps.set_ground_plane_mode("shadow_only")
    ps.set_window_size(width, height)

    # 锁定绝对物理尺度
    ps.set_automatically_compute_scene_extents(False)
    ps.set_length_scale(1.0)
    ps.set_bounding_box((-1.5, -1.5, -0.5), (1.5, 1.5, 1.5))

    video_frames = []
    tmp_img_path = "tmp_neon_meteor.jpg"

    # 3. 4D 影视级渲染循环
    for t in tqdm(range(n_frames), desc=f"🎥 4D 霓虹流星 (Polyscope)"):
        # --- A. 计算极度平滑的相机轨迹 (Orbit: cam_start -> cam_end) ---
        alpha = t / max(1, n_frames - 1)
        # 提取起止相机的 Eye (位置) 和 Forward (Z轴朝向)
        eye_start = scene_state[cam_start]['extrinsics'][t][:3, 3]
        eye_end = scene_state[cam_end]['extrinsics'][t][:3, 3]
        z_start = scene_state[cam_start]['extrinsics'][t][:3, 2]
        z_end = scene_state[cam_end]['extrinsics'][t][:3, 2]

        eye_interp = (1 - alpha) * eye_start + alpha * eye_end
        target_interp = eye_interp + ((1 - alpha) * z_start + alpha * z_end)

        # Pullback 运镜：让镜头顺着视线往后拉远一点，获得更宏大的视野
        look_dir = target_interp - eye_interp
        look_dir /= (np.linalg.norm(look_dir) + 1e-6)
        eye_interp -= look_dir * pullback_dist

        ps.look_at(tuple(eye_interp), tuple(target_interp))

        # --- B. 缝合静态环境 (并人为压暗亮度，营造“暗场”效果) ---
        env_results = [
            unproject_to_3d(
                depth=scene_constants['camera'][cam]['raw_depth'][t].astype(np.float32),
                color_img=scene_constants['camera'][cam]['video_rgb'][t].astype(np.float32) / 255.0,
                K_mat=scene_constants['camera'][cam]['K_mat'],
                T_cam2world=scene_state[cam]['extrinsics'][t],
                min_depth=min_depth, max_depth=max_depth
            ) for cam in camera_ids
        ]
        env_pts, env_cols = map(np.vstack, zip(*env_results))

        if len(env_pts) > 0:
            idx = np.random.choice(len(env_pts), min(max_env_points, len(env_pts)), replace=False)
            # 🌟 核心技巧：将环境光降低到 35%，充当赛博朋克的暗夜背景！
            dimmed_cols = np.clip(env_cols[idx] * 0.35, 0, 1)
            ps_env = ps.register_point_cloud("env", env_pts[idx], radius=0.002, point_render_mode='sphere')
            ps_env.add_color_quantity("rgb", dimmed_cols, enabled=True)

        # --- C. 构建 3D 霓虹圆管拖尾 (Curve Network) ---
        start_t = max(0, t - tail_length)
        all_nodes, all_edges, all_node_colors = [], [], []
        node_offset = 0

        for i in range(traj_3d.shape[1]):
            window_valid = traj_valid[start_t:t+1, i]
            if not window_valid.any(): continue

            valid_times = np.where(window_valid)[0]
            actual_times = start_t + valid_times
            pts = traj_3d[actual_times, i]
            n_pts = len(pts)

            if n_pts > 0:
                all_nodes.append(pts)
                # 计算透明度 Fade：越早的轨迹越暗
                fade = (actual_times - start_t) / max(1, t - start_t)
                fade_colors = point_colors[i] * fade[:, None]
                all_node_colors.append(fade_colors)

            # 把这一个点的历史轨迹连成线 (edges)
            if n_pts > 1:
                edges = np.column_stack((np.arange(n_pts - 1), np.arange(1, n_pts))) + node_offset
                all_edges.append(edges)
            node_offset += n_pts

        if all_nodes:
            cat_nodes = np.vstack(all_nodes)
            cat_cols = np.vstack(all_node_colors)
            cat_edges = np.vstack(all_edges) if all_edges else np.empty((0, 2), dtype=int)

            if len(cat_edges) > 0:
                # 🌟 注册为真 3D 圆管，并使用 flat 无光照材质发出霓虹光！
                ps_net = ps.register_curve_network("neon_tails", cat_nodes, cat_edges, radius=0.0025, material='flat')
                ps_net.add_color_quantity("fade_colors", cat_cols, defined_on='nodes', enabled=True)

        # --- D. 绘制流星头部 (明亮的球体) ---
        current_valid = traj_valid[t]
        if current_valid.any():
            head_pts = traj_3d[t, current_valid]
            head_cols = point_colors[current_valid]
            # 头部半径略大于拖尾，同样使用纯粹的 flat 材质
            ps_heads = ps.register_point_cloud("neon_heads", head_pts, radius=0.004, point_render_mode='sphere', material='flat')
            ps_heads.add_color_quantity("head_colors", head_cols, enabled=True)

        # --- E. 极速物理落盘与盖水印 ---
        ps.screenshot(tmp_img_path, transparent_bg=False)

        if os.path.exists(tmp_img_path):
            img_rgb = media.read_image(tmp_img_path).copy()
            watermark = f"4D Neon Flow | Src: {src_cam} | Orbit: {cam_start} -> {cam_end}"
            cv2.putText(img_rgb, watermark, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
            cv2.putText(img_rgb, watermark, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            video_frames.append(img_rgb)

        ps.remove_all_structures()

    return video_frames

# =========================================================
# 主数据重构流水线：重新提取双轨白金真值并输送给渲染器
# =========================================================
camera_keys = list(sliced_scene_constants['camera'].keys())
target_src_cam = camera_keys[1] if len(camera_keys) > 1 else camera_keys[0]

print(f"🔄 正在为 4D 渲染重构 3D 白金真值 (主视角: {target_src_cam})...")

T_frames = len(sliced_scene_constants['camera'][target_src_cam]['video_rgb'])
cam_data = sliced_scene_constants['camera'][target_src_cam]
cam_state = sliced_scene_state[target_src_cam]
h_img, w_img = cam_data['video_rgb'][0].shape[:2]
tracks_2d = cam_data['tracks_2d']
raw_depth = cam_data['raw_depth']

# 确保双引擎就绪
if 'pb_renderer_ultimate' not in locals():
    pb_renderer_ultimate = PyBulletRenderer_Robotiq()
if 'urdf_tracker' not in locals():
    urdf_tracker = URDFKinematicsTracker(pb_renderer_ultimate)
if 'consensus_tracker' not in locals():
    consensus_tracker = ConsensusVisualTracker(cotracker_model, device)

# ---------------------------------------------------------
# 🦾 1. 提取 Track B (URDF 物理刚体点)
# ---------------------------------------------------------
traj_3d_rob, proj_2d_rob, vis_rob, robot_indices = urdf_tracker.extract_robot_tracks(
    target_src_cam, sliced_scene_constants, sliced_scene_state
)
if proj_2d_rob is not None:
    orig_2d_rob = tracks_2d[:, robot_indices, :]
else:
    traj_3d_rob = np.empty((T_frames, 0, 3), dtype=np.float32)
    vis_rob = np.empty((T_frames, 0), dtype=bool)
    orig_2d_rob = np.empty((T_frames, 0, 2), dtype=np.float32)

# ---------------------------------------------------------
# 👁️ 2. 提取 Track A (15票共识环境点 + 终极漏斗过滤)
# ---------------------------------------------------------
joint_angles_t0 = sliced_scene_constants['robot']['joint_positions'][0]
gripper_state_t0 = sliced_scene_constants['robot']['gripper_positions'][0]
pb_renderer_ultimate.update_robot_pose(joint_angles_t0, gripper_state=gripper_state_t0)

robot_mask = pb_renderer_ultimate.render_mask(cam_state['extrinsics'][0], cam_data['K_mat'], w_img, h_img) > 0
kernel = np.ones((15, 15), np.uint8)
robot_mask_dilated = cv2.dilate(robot_mask.astype(np.uint8), kernel, iterations=1) > 0

u0 = np.clip(np.round(tracks_2d[0, :, 0]).astype(int), 0, w_img-1)
v0 = np.clip(np.round(tracks_2d[0, :, 1]).astype(int), 0, h_img-1)
is_env_point = ~robot_mask_dilated[v0, u0]
has_valid_depth = (raw_depth[0, v0, u0] > 0.05) & (raw_depth[0, v0, u0] < 5.0)
test_env_indices = np.where(is_env_point & has_valid_depth)[0]

smoothed_3d, refined_2d, unified_vis = consensus_tracker.extract_env_tracks(
    target_src_cam, test_env_indices, sliced_scene_constants, sliced_scene_state
)

original_2d_env_raw = tracks_2d[:, test_env_indices, :]
valid_counts = np.sum(unified_vis, axis=0)
D_2D = np.linalg.norm(original_2d_env_raw - refined_2d[:, :, :2], axis=-1)
max_drift = np.max(np.where(unified_vis, D_2D, 0), axis=0)
flicker_counts = np.sum(unified_vis[:-1] != unified_vis[1:], axis=0)

survivor_mask = (valid_counts >= 5) & (max_drift < 10.0) & (flicker_counts <= T_frames * 0.10)
golden_indices = np.where(survivor_mask)[0]

if len(golden_indices) > 0:
    traj_3d_env = smoothed_3d[:, golden_indices, :]
    vis_env = unified_vis[:, golden_indices].copy()
    orig_2d_env = original_2d_env_raw[:, golden_indices, :]
    proj_2d_env = refined_2d[:, golden_indices, :2]
    z_pred_env = refined_2d[:, golden_indices, 2]

    # 附加 X 光透视物理遮挡修复
    for t in range(T_frames):
        ui = np.clip(np.round(proj_2d_env[t, :, 0]).astype(int), 0, w_img - 1)
        vi = np.clip(np.round(proj_2d_env[t, :, 1]).astype(int), 0, h_img - 1)
        z_sensor = raw_depth[t, vi, ui]
        xray_occluded = (z_sensor > 0) & (z_sensor < z_pred_env[t] - 0.05)
        vis_env[t] = vis_env[t] & (~xray_occluded)
else:
    traj_3d_env = np.empty((T_frames, 0, 3), dtype=np.float32)
    vis_env = np.empty((T_frames, 0), dtype=bool)
    orig_2d_env = np.empty((T_frames, 0, 2), dtype=np.float32)

# ---------------------------------------------------------
# 🤝 3. 双轨合并并生成调色板，送入 4D 渲染器
# ---------------------------------------------------------
combined_traj_3d = np.concatenate([traj_3d_rob, traj_3d_env], axis=1)
combined_vis = np.concatenate([vis_rob, vis_env], axis=1)
combined_orig_2d = np.concatenate([orig_2d_rob, orig_2d_env], axis=1)

# 生成色彩矩阵
y_coords = combined_orig_2d[0, :, 1]
if len(y_coords) > 1 and y_coords.max() > y_coords.min():
    norm = plt.Normalize(y_coords.min(), y_coords.max())
else:
    norm = plt.Normalize(y_coords.min() - 1, y_coords.min() + 1)
point_colors = plt.cm.gist_rainbow(norm(y_coords))[:, :3]

# 启动引擎！
output_video = render_4d_rainbow_meteor_flow(
    scene_constants=sliced_scene_constants,
    scene_state=sliced_scene_state,
    src_cam=target_src_cam,
    traj_3d=combined_traj_3d,        # 🌟 直接注入合并后的纯净 3D 轨迹
    traj_valid=combined_vis,         # 🌟 直接注入合并后的生命周期掩码
    point_colors=point_colors,       # 🌟 注入对应的彩虹配色
    max_env_points=400000
)

if len(output_video) > 0:
    media.show_video(output_video, fps=15, codec='gif')

### 生成Droid数据集

In [ ]:
# @title 🚀 单轨 URDF 物理点提纯版

import os
import cv2
import numpy as np

def extract_and_export_tapvid(export_root, scene_constants, scene_state, urdf_tracker, consensus_tracker=None, pb_renderer=None):
    """
    精简版：仅提取 URDF 物理刚体 3D 轨迹 (Track B) 并序列化为 TAPVid 格式
    注：保留 consensus_tracker 和 pb_renderer 参数仅为兼容外部批处理脚本的调用，函数内部不再使用。
    """
    print("  🧹 启动单轨 (URDF) 提取、跨视角重投影与 TAPVid 最终落盘...")
    episode_id = scene_constants['meta']['episode_id']
    camera_ids = list(scene_constants['camera'].keys())
    T_frames = len(scene_constants['camera'][camera_ids[0]]['video_rgb'])

    all_final_tracks = []
    all_queries = []
    all_visibility = {cam: [] for cam in camera_ids}
    total_survivors = 0

    # 🌟 1. 遍历每一个机位作为“主视角 (Source View)”
    for v_source_idx, src_cam in enumerate(camera_ids):
        print(f"    🔍 正在提取视角 [{src_cam}] 的 URDF 刚体轨迹...")
        cam_data = scene_constants['camera'][src_cam]
        h_img, w_img = cam_data['video_rgb'][0].shape[:2]
        tracks_2d = cam_data['tracks_2d']

        orig_2d_all = []
        traj_3d_all = []
        vis_all = []

        # ==========================================
        # 🦾 Track B: 提取 URDF 机械臂纯物理点
        # ==========================================
        traj_3d_rob, proj_2d_rob, vis_rob, robot_indices = urdf_tracker.extract_robot_tracks(
            src_cam, scene_constants, scene_state
        )

        if proj_2d_rob is not None and len(robot_indices) > 0:
            orig_2d_all.append(tracks_2d[:, robot_indices, :])
            traj_3d_all.append(traj_3d_rob)
            vis_all.append(vis_rob)

        # ==========================================
        # 🤝 汇总检查
        # ==========================================
        if not orig_2d_all:
            print(f"    ⚠️ 视角 [{src_cam}] 未保留下任何 URDF 点位。")
            continue

        combined_orig_2d = np.concatenate(orig_2d_all, axis=1) # [T, N, 2]
        combined_traj_3d = np.concatenate(traj_3d_all, axis=1) # [T, N, 3]
        combined_vis = np.concatenate(vis_all, axis=1)         # [T, N]

        N_survivors = combined_traj_3d.shape[1]
        total_survivors += N_survivors
        all_final_tracks.append(combined_traj_3d)

        # 收集 Queries (对应初始帧)
        queries = np.zeros((N_survivors, 4), dtype=np.float32)
        queries[:, 0] = combined_orig_2d[0, :, 0]
        queries[:, 1] = combined_orig_2d[0, :, 1]
        queries[:, 2] = 0                      # Frame=0
        queries[:, 3] = v_source_idx           # Source Cam Index
        all_queries.append(queries)

        # ==========================================
        # 🎯 Target 视角交叉投影与 X 光遮挡解算
        # ==========================================
        for tgt_cam in camera_ids:
            if tgt_cam == src_cam:
                all_visibility[tgt_cam].append(combined_vis)
                continue

            tgt_data = scene_constants['camera'][tgt_cam]
            tgt_state = scene_state[tgt_cam]
            tgt_raw_depth = tgt_data['raw_depth']
            tgt_K = tgt_data['K_mat']
            tgt_ext = tgt_state['extrinsics']

            tgt_vis = np.zeros((T_frames, N_survivors), dtype=bool)

            for t in range(T_frames):
                valid_3d = ~np.isnan(combined_traj_3d[t, :, 2])
                if valid_3d.any():
                    u_v, v_v, z_p = project_points_np(combined_traj_3d[t, valid_3d], tgt_K, tgt_ext[t])
                    ui = np.clip(np.round(u_v).astype(int), 0, w_img - 1)
                    vi = np.clip(np.round(v_v).astype(int), 0, h_img - 1)

                    in_bounds = (u_v >= 0) & (u_v < w_img) & (v_v >= 0) & (v_v < h_img) & (z_p > 0)
                    z_sensor = tgt_raw_depth[t, vi, ui]

                    # Target 视角的 X光透视屏蔽
                    xray_occluded = (z_sensor > 0) & (z_sensor < z_p - 0.05)
                    tgt_vis_t = np.zeros(N_survivors, dtype=bool)
                    tgt_vis_t[valid_3d] = in_bounds & (~xray_occluded)
                    tgt_vis[t] = tgt_vis_t

            all_visibility[tgt_cam].append(tgt_vis)

    if total_survivors == 0:
        print(f"  ⚠️ 警告：所有视角均无有效 URDF 点位！跳过落盘。")
        return

    # 🌟 2. 大一统拼接
    final_tracks_xyz = np.concatenate(all_final_tracks, axis=1) # [T, N_total, 3]
    queries_xytv = np.concatenate(all_queries, axis=0)          # [N_total, 4]

    # 🌟 3. 全局文件落盘
    seq_dir = os.path.join(export_root, episode_id)
    os.makedirs(seq_dir, exist_ok=True)

    np.save(os.path.join(seq_dir, "tracks_xyz.npy"), final_tracks_xyz.astype(np.float32))
    np.save(os.path.join(seq_dir, "queries_xytv.npy"), queries_xytv)

    # 🌟 4. 各个 Target View 的底层数据分别落盘
    for v_idx, cam in enumerate(camera_ids):
        cam_dir = os.path.join(seq_dir, str(v_idx))
        os.makedirs(cam_dir, exist_ok=True)
        cam_data = scene_constants['camera'][cam]

        # 视频帧 JPEG 字节流化
        jpeg_bytes_list = [cv2.imencode('.jpg', cv2.cvtColor(img, cv2.COLOR_RGB2BGR))[1].tobytes() for img in cam_data['video_rgb']]
        np.save(os.path.join(cam_dir, "images_jpeg_bytes.npy"), np.array(jpeg_bytes_list, dtype=object))

        # 将这个目标相机对【全部源视角点云】的可见性横向拼接成 [T, N_total]
        final_vis = np.concatenate(all_visibility[cam], axis=1)
        np.save(os.path.join(cam_dir, "visibility.npy"), final_vis)

        # 相机内外参落盘
        K = cam_data['K_mat']
        np.save(os.path.join(cam_dir, "intrinsics.npy"), np.array([K[0,0], K[1,1], K[0,2], K[1,2]], dtype=np.float32))
        np.save(os.path.join(cam_dir, "extrinsics_w2c.npy"), np.linalg.inv(scene_state[cam]['extrinsics']).astype(np.float32))

    print(f"  🎉 提纯成功！三视角共保留 {total_survivors} 个 URDF 物理真值点，已存储至: {seq_dir}")

In [ ]:
# @title 🚀 全自动批处理引擎 (单轨 URDF 物理点提纯版 - GCS 直读提速版)
import os
import gc
import cv2
import numpy as np
import traceback

# ================= 黄金目标池 =================
valid_ids = [
    "GuptaLab+553d1bd5+2023-04-30-16h-07m-27s",
    "GuptaLab+553d1bd5+2023-05-28-16h-48m-13s",
    "GuptaLab+553d1bd5+2023-05-28-18h-22m-39s",
    "ILIAD+5e938e3b+2023-07-20-11h-50m-51s",
    "ILIAD+7ae1bcff+2023-06-04-20h-13m-12s",
    "IPRL+edf28ef3+2024-01-01-10h-36m-18s",
    "IRIS+7dfa2da3+2023-04-17-16h-24m-37s",
    "IRIS+7dfa2da3+2023-05-11-14h-12m-57s",
    "IRIS+7dfa2da3+2023-06-01-13h-48m-16s",
    "PennPAL+c5f808b7+2023-06-15-17h-12m-39s",
    "RAIL+d027f2ae+2023-11-04-14h-49m-32s",
    "REAL+4f8ca688+2023-08-29-15h-01m-27s",
    "REAL+75b7b0f9+2023-06-23-16h-46m-08s",
    "REAL+de601749+2023-12-19-18h-19m-27s",
    "TRI+52ca9b6a+2023-12-05-15h-19m-45s",
    "TRI+52ca9b6a+2024-01-23-16h-53m-54s",
]

export_root = "/content/DROID-mv-eval"
os.makedirs(export_root, exist_ok=True)

# 目标：随机/末尾 截取 48 帧
SLICE_FRAMES = 48
TARGET_EPISODES = valid_ids

print("\n" + "="*60)
print("🚀 启动全自动批处理流水线 (单轨 URDF 物理轨迹提纯版 - GCS极速直读)")
print("="*60)

# 🌟 提前初始化物理引擎与 URDF 引擎
pb_renderer_ultimate = PyBulletRenderer_Robotiq()
urdf_tracker = URDFKinematicsTracker(pb_renderer_ultimate)

for ep_idx, episode_id in enumerate(TARGET_EPISODES):
    print(f"\n🎬 [Episode {ep_idx+1}/{len(TARGET_EPISODES)}] 开始处理: {episode_id}")

    scene_constants, scene_state = None, None

    try:
        # 🌟 1. 基础数据骨架初始化
        scene_constants = download_episode(episode_id, root_path, id_to_path, serials_db, keep_ranges)

        # 🌟 2. 【核心提速】直接从 GCS Bucket 拉取处理好的深度图、视频、机器臂关节信息等
        scene_constants = load_stage1_camera_data(scene_constants)

        # 🌟 3. 【核心提速】从 GCS Bucket 直接拉取完美的 4x4 外参大盘
        scene_state = load_stage2_extrinsics_json(scene_constants)

        # 🌟 4. 双轨联动：剔除静止发呆帧
        scene_constants, scene_state = filter_idle_frames(scene_constants, scene_state)

        # 如果剔除后长度不足，跳过
        if len(scene_constants['robot']['joint_positions']) < SLICE_FRAMES:
            print("  ⏭️ 视频有效动作太短，跳过。")
            continue

        # 🌟 5. 时序截断 (截取 48 帧)
        scene_constants, scene_state = slice_scene_temporal_window(
            scene_constants, scene_state, n=SLICE_FRAMES, mode='random'
        )

        # 🌟 6. 执行 CoTracker 2D 追踪 (必须跑，因为 URDF 需要利用第0帧的稠密点作为抓取种子)
        scene_constants = extract_2d_tracks(cotracker_model, scene_constants)

        # 🌟 7. 单轨 URDF 提取与 TAPVid 格式落盘
        extract_and_export_tapvid(
            export_root=export_root,
            scene_constants=scene_constants,
            scene_state=scene_state,
            urdf_tracker=urdf_tracker,
            consensus_tracker=None,
            pb_renderer=pb_renderer_ultimate
        )

    except Exception as e:
        print(f"  ❌ 处理该集时发生意外错误: {e}")
        traceback.print_exc()

    finally:
        # 🌟 8. 滴水不漏的显存回收机制
        del scene_constants, scene_state
        gc.collect()
        torch.cuda.empty_cache()

print("\n🏆 所有指定视频极速批处理完成！请移步验证单元格查看！")

In [ ]:
# @title ☁️ 自动打包并上传到 Google Drive

import os
import shutil
from google.colab import drive

# 🌟 1. 挂载云端硬盘 (建立跨界通道)
print("🔗 正在请求挂载 Google Drive (此时会弹出一个授权窗口，请点击允许)...")
drive.mount('/content/drive')

# 🌟 2. 语义化路径指针：杜绝硬编码，让数据的流向一目了然
source_dataset_dir = "/content/DROID-mv-eval"
local_zip_path = "/content/DROID_v2.zip"
drive_target_dir = "/content/drive/MyDrive/MVTAPVid3D"

os.makedirs(drive_target_dir, exist_ok=True)

# 🌟 3. 纯粹的 Python 极速打包：利用 shutil 替代生硬杂乱的 Shell 命令
print("\n📦 正在将高纯度数据集打包为 ZIP (开启极速传输模式)...")
# make_archive 极其优雅，会自动补充 .zip 后缀，且在底层执行最高效的流式压缩
shutil.make_archive(local_zip_path.replace('.zip', ''), 'zip', source_dataset_dir)

# 🌟 4. 平滑跨界转移：安全落盘至 Google Drive
print(f"\n🚀 正在将压缩包转移至你的云端硬盘: {drive_target_dir} ...")
shutil.copy2(local_zip_path, drive_target_dir)
print("✅ 备份完美收官！压缩包已安全躺在你的 Google Drive 中。")

# 🌟 5. 终极打扫战场：滴水不漏地释放 Colab 宝贵的本地空间
print("\n🧹 正在清理 Colab 本地缓存，释放硬盘空间...")
shutil.rmtree(source_dataset_dir, ignore_errors=True)
if os.path.exists(local_zip_path):
    os.remove(local_zip_path)
print("✨ 本地空间已清空，随时可以开始下一批次的提纯！")

In [ ]:
# @title 📊 Read saved data and execute ultimate visual validation (includes 4D interactive point cloud player)
import os
import shutil
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
import mediapy as media
import plotly.graph_objects as go

zip_path = "/content/drive/MyDrive/MVTAPVid3D/DROID_v2.zip"
extract_path = "/content/DROID-mv-eval"

if os.path.exists(zip_path):
    print(f"📦 Rescuing data from Google Drive, extracting to {extract_path}...")
    shutil.unpack_archive(zip_path, extract_path)
    print("✅ Rescue successful! You can now re-run the [Visual Validation] cell.")
else:
    print("⚠️ Backup zip file not found, please check the path.")

export_root = "/content/DROID-mv-eval"
saved_episodes = [d for d in os.listdir(export_root) if os.path.isdir(os.path.join(export_root, d))]

# ==========================================
# 🌟 New: 4D Interactive Point Cloud Player Core Operator
# ==========================================
def show_animated_plotly_point_cloud(traj_3d, colors_rgb, title="Animated 3D Tracks", eye_pos=(0, -0.8, -1.5)):
    """Dynamic interactive 3D point cloud player, supports timeline dragging and playback"""
    T, N, _ = traj_3d.shape

    # 1. Convert colors to rgb strings supported by Plotly
    hex_colors = [f'rgb({int(r)},{int(g)},{int(b)})' for r, g, b in colors_rgb]

    # 2. Lock the global physical coordinate system boundaries to prevent wild shaking during playback
    x_min, x_max = np.nanmin(traj_3d[:, :, 0]), np.nanmax(traj_3d[:, :, 0])
    y_min, y_max = np.nanmin(traj_3d[:, :, 1]), np.nanmax(traj_3d[:, :, 1])
    z_min, z_max = np.nanmin(traj_3d[:, :, 2]), np.nanmax(traj_3d[:, :, 2])

    # 3. Build the base canvas for frame 0
    fig = go.Figure(
        data=[go.Scatter3d(
            x=traj_3d[0, :, 0], y=traj_3d[0, :, 1], z=traj_3d[0, :, 2],
            mode='markers', marker=dict(size=2.0, color=hex_colors)
        )]
    )

    # 4. Push in the skeleton data for all time steps
    frames = []
    for t in range(T):
        frames.append(go.Frame(
            data=[go.Scatter3d(x=traj_3d[t, :, 0], y=traj_3d[t, :, 1], z=traj_3d[t, :, 2])],
            name=str(t)
        ))
    fig.frames = frames

    # 5. Assemble the player panel and timeline
    sliders = [dict(
        steps=[dict(method='animate',
                    args=[[str(t)], dict(mode='immediate', frame=dict(duration=80, redraw=True), transition=dict(duration=0))],
                    label=f"F{t}") for t in range(T)],
        active=0, transition=dict(duration=0), x=0, y=0
    )]

    fig.update_layout(
        title=title,
        margin=dict(l=0, r=0, b=0, t=40),
        height=600, showlegend=False,
        scene=dict(
            aspectmode='data',
            xaxis=dict(range=[x_min, x_max], autorange=False),
            yaxis=dict(range=[y_min, y_max], autorange=False),
            zaxis=dict(range=[z_min, z_max], autorange=False),
            camera=dict(eye=dict(x=eye_pos[0], y=eye_pos[1], z=eye_pos[2]))
        ),
        updatemenus=[dict(
            type="buttons", showactive=False, x=0.05, y=1.1,
            buttons=[
                dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=80, redraw=True), transition=dict(duration=0), fromcurrent=True)]),
                dict(label="⏸ Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate", transition=dict(duration=0))])
            ]
        )],
        sliders=sliders
    )
    fig.show(renderer="colab")

# ==========================================
# 🌟 Main Acceptance Flow
# ==========================================
if not saved_episodes:
    print("⚠️ No saved data found, please run the previous fully automatic batch engine first!")
else:
    # 🌟 1. Randomly sample a saved Episode
    test_ep = random.choice(saved_episodes)
    print(f"\n🔍 Reading and validating saved data: {test_ep}")
    ep_dir = os.path.join(export_root, test_ep)

    # 🌟 2. Extract 3D trajectory ground truth
    traj_3d = np.load(os.path.join(ep_dir, "tracks_xyz.npy")) # [T, N, 3]
    n_frames, n_points, _ = traj_3d.shape
    print(f"  ✅ Successfully read 3D trajectory! Frames: {n_frames}, Premium tracking points: {n_points}")

    # 🌟🌟🌟 Core fix: Directly read Query coordinates as the globally unique palette reference 🌟🌟🌟
    queries = np.load(os.path.join(ep_dir, "queries_xytv.npy"))
    y_vals_src = queries[:, 1] # Extract the Y coordinate of the initial frame of the source view
    global_colors = plt.cm.gist_rainbow(plt.Normalize(y_vals_src.min(), y_vals_src.max())(y_vals_src))[:, :3] * 255

    # Get all camera views (automatically identify 0, 1, 2 folders)
    cam_dirs = sorted([d for d in os.listdir(ep_dir) if d.isdigit()], key=int)
    all_grid_videos = []

    for cam_idx in cam_dirs:
        cam_path = os.path.join(ep_dir, cam_idx)
        print(f"  🎥 Reconstructing matrices and frames for view [{cam_idx}]...")

        # Read core matrices and visibility
        visibility = np.load(os.path.join(cam_path, "visibility.npy")) # [T, N]
        intrinsics = np.load(os.path.join(cam_path, "intrinsics.npy")) # [fx, fy, cx, cy]
        extrinsics_w2c = np.load(os.path.join(cam_path, "extrinsics_w2c.npy")) # [T, 4, 4]

        # Elegantly restore K matrix
        K_mat = np.array([
            [intrinsics[0], 0, intrinsics[2]],
            [0, intrinsics[1], intrinsics[3]],
            [0, 0, 1]
        ])

        # Restore camera poses in the world coordinate system (T_cam2world)
        extrinsics_c2w = np.linalg.inv(extrinsics_w2c)

        # Instantly decode JPEG byte stream
        jpeg_bytes = np.load(os.path.join(cam_path, "images_jpeg_bytes.npy"), allow_pickle=True)
        video_frames = []
        for b in jpeg_bytes:
            img_bgr = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_COLOR)
            video_frames.append(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

        # 🌟 3. Project the 3D trajectory back onto the 2D plane
        proj_trk = np.zeros((n_frames, n_points, 2))
        for t in range(n_frames):
            u, v, _ = project_points_np(traj_3d[t], K_mat, extrinsics_c2w[t])
            proj_trk[t, :, 0] = u
            proj_trk[t, :, 1] = v

        # 🌟 4. Call the original drawing module and force inject the newly calculated global_colors!
        rendered_frames = render_2d_tracking_video(
            video_frames=video_frames,
            tracks=proj_trk,
            visibility=visibility,
            global_colors=global_colors, # <--- Colors are synchronized here!
            linewidth=4,
        )

        # Add exclusive acceptance stamp and align scaling
        cam_vid = []
        for img in rendered_frames:
            text = f"View {cam_idx} Re-Proj"
            cv2.putText(img, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
            cv2.putText(img, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
            cam_vid.append(cv2.resize(img, (320, int(320 * img.shape[0] / img.shape[1]))))

        all_grid_videos.append(np.array(cam_vid))

    # 🌟 5. Horizontally concatenate and screen
    print("\n🎨 2D re-projection quality check complete! The rainbow meteors from the three views are now absolutely synchronized!")
    media.show_video(np.concatenate(all_grid_videos, axis=2), fps=15, codec='gif')

    # 🌟 6. Bonus 3D sparse skeleton booth (dynamic playback version)
    print("\n🌌 Generating saved 4D point cloud interactive array for you (supports free rotation and timeline dragging)...")

    # Core: Feed the full-time sequence traj_3d into the new dynamic operator
    show_animated_plotly_point_cloud(
        traj_3d=traj_3d,
        colors_rgb=global_colors,
        title=f"4D Sparse Tracks - {test_ep}",
        eye_pos=(0, -0.8, -1.5)
    )